# 02 — Matched NVIDIA RGB/Event Source: D200 Expansion

**Fresh Colab T4 runtime → Run All.** Drive authorization is expected. No training, policy weights or scored episodes are used. Keep existing D50 datasets/checkpoints/results in their current Drive locations.

## Purpose and locked split roles

Prepare all 200 recorded demonstrations with the source lifecycle already verified for D50. The existing D50 40 training and 10 validation roles are preserved exactly. The additional 50 demos in D100 add 40 train/10 validation; the remaining 100 demos add 80 train/20 validation. Result: **160 training / 40 validation**, with the original ten held-out D50 demos retained as a common validation subset. Only split seed20260927 determines selection/roles; policy seeds42/43/44 and final rollout seeds1000–1049 remain separate.

The notebook validates `configs/splits/d50.json`, `d100.json`, `d200.json` against Notebook01's exact seeded selection order and the locked D50 role file. It creates `configs/training/d200_rgb_event_train_val.json` deterministically once. An existing different D200 role file is protected and blocks execution; it is never silently overwritten.

## Source contract preserved

- Each demo uses its own recorded XML and immutable state/action sequence. Restore/forward/render; no demonstration action replay. Intermediate source states use the same 200Hz linear/SLERP reconstruction of 20Hz recorded states. They are approximations, not measured high-rate physics.
- Two cameras, top-origin84×84. Three fixed native reads; choose draw3. Draw2/3 grayscale must match exactly, RGB differences stay within the existing one-level/two-color-value bound. No looser renderer gate, filtering or retries.
- Persistent per-camera CPU V2E is interleaved with NVIDIA rendering. The same pinned commit, thresholds0.2, zero noise/leak and preprocessing seed42 are retained. Raw Event timestamps/window bounds remain float32 half-open intervals; tensors remain positive/negative log1p counts plus tau0.05 time surface.
- Robot-only18D per observation /36D total, first policy sample at step2, observations(t-1,t), action targets[t:t+16]. No sample crosses a demo boundary.
- The pinned V2E settings are copied from the verified demo10 source into a separate D200 settings namespace. Newly added demos do not need absent historical `data/processed/events/<demo>/manifest.json` files. These settings files are configuration, not claims that those demos were already processed.
- Both runtime binding and the actual V2E initializer read that same explicit settings file and verify its SHA256. The historical initializer lookup that caused the first new D200 demo to fail in v2 is removed administratively; the numerical emulator and renderer remain unchanged.

## Automatic sequence, reuse and resume

1. Verify raw hash, all200 raw dimensions/timestamps, split files and the exact reviewed D50 source protocol. Install the already exercised simulator/V2E recipe.
2. Probe actual NVIDIA hardware EGL in fresh subprocesses. Compare driver, graphics library, dependency stack, Python and source contracts to D50.
3. If fully compatible, reference the fifty existing D50 sources read-only after per-demo hash, role, own-scene, state/action and acquisition checks. Full sources are not copied or relabeled as newly generated data. If runtime identities differ, prepare all200 in the current runtime in a new dataset namespace instead. No silent mixing or software-renderer fallback.
4. Run a mandatory independent causal-source/noninterference/CPU-vs-GPU physics preflight on training demo10 in this runtime. A reused demo10 is compared using its original provenance, not forged D200 provenance. Stop before the remaining demos if this fails.
5. Generate the other required sources sequentially in fresh per-demo workers; retain the verified numerical worker unchanged. Publish a completion binding only after validation. Run All resumes verified bindings and gives incomplete attempts new generation folders. A failed source blocks completion; no demo is silently dropped.
6. Publish `configs/processed_sources/paired_gpu_d200_latest.json` only after all200 pass. The actual frame budget is derived from all raw lengths; an explicit hard cap blocks oversized data rather than truncating it.

## Paths and resource use

New dataset: `data/processed/paired_gpu_source/d200_paired_gpu_v2_<protocol_hash>/`. Historical D50 references are explicit, hashed and confined to the exact reviewed D50 root. Those original files must remain available. No old source index, normalizer, training run or checkpoint is overwritten.

**T4 is required for Notebook02 hardware rendering; V2E itself runs on CPU.** When compatibility passes, only the150 additional demos need new bulk source generation, plus the mandatory current-runtime preflight. This is a conditional saving, not a promised runtime or compute-unit estimate.

After completion upload `paired_source_build_bundle.zip` from the printed session path under `results/paired_source_preparation/notebook02/sessions/<session>/`. Then switch to a CPU runtime and run Notebook03. On failure, upload the same session bundle before proceeding. The bundle excludes large source arrays; they remain on Drive.

All text/comments are English. D200 CUDA/EGL/V2E execution is not yet verified at delivery; this Run All supplies that evidence. Additional data is an experiment, not a guaranteed cure for Event task performance. No policy training starts here.


## 1. Central configuration and Drive connection


In [ ]:
# Central configuration. Data preparation only; no training or scored rollouts.
PROJECT_DIRECTORY='/content/drive/MyDrive/DiffusionPolicyEvents'
STAGE='02'
CODE_VERSION='paired-gpu-source-d200-v3'
BUILD_SHA='0dba021949ead3a91c4c40bf9ffe20e79b067061ae7c202c5e3188d09c28ae2d'
OPTIONS={'dataset':'d200','nvidia_library':'/usr/lib64-nvidia/libEGL_nvidia.so.0',
    'hard_max_frames_per_camera':10001,'source_timeout_s':900,
    'training_seed_roles':[42,43,44],'split_seed_role':20260927,'event_preprocessing_seed':42}
assert OPTIONS['dataset']=='d200'
import ast,base64,hashlib,importlib,importlib.util,importlib.metadata,json,os,shutil,subprocess,sys,tempfile,time,traceback,uuid,zlib
from pathlib import Path
from datetime import datetime,timezone
from google.colab import drive
drive.mount('/content/drive')
PROJECT_ROOT=Path(PROJECT_DIRECTORY)
assert (PROJECT_ROOT/'data/raw/lift_ph_raw.hdf5').is_file(), 'Verified immutable raw dataset required.'
SESSION=PROJECT_ROOT/'results/paired_source_preparation'/('notebook'+STAGE)/'sessions'/(datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'_'+uuid.uuid4().hex[:8])
SESSION.mkdir(parents=True,exist_ok=False)
WORKSPACE=Path(tempfile.mkdtemp(prefix='nb'+STAGE+'-paired-d200-',dir='/content'))
VALIDATION_BUNDLE=PROJECT_ROOT/'data/processed/gpu_source_pilots/nvidia_three_demo_a9275985e814/sessions/20261007T111758Z_b875cb90/gpu_source_validation_bundle.zip'
print('Notebook',STAGE,'| D200 preparation only | evidence:',SESSION)


## 2. Embedded source, dependencies and complete pipeline definitions


In [ ]:
COMMON_TEXT=zlib.decompress(base64.b64decode('eNqtO2tv20iS3/0reDlgSI0phqSeVqIBjMSzFyCTBIlvFlhBIFpkU+KaryUp2XJu/vtVVXfzJcmbGawxGLFf1dX1rurOq1evfudFGWUpD7ScRQUPhmW2L3yu5dhfVjyFb5YGWrpPeBH5LNb8LK0K5lelpX3KtDyLI/+oQU+URunWevXq1VWU5FlRaX55MLUdK3dxtDG1KDO1f5ZZampZaWrlbl9FsalVUcJNbb+PAlN7jvIwivlVWGQJYFPhOk3C+gJNMRCwiuMqNaLaAtYzHEXtDyjnR42VWppfXX29/bv37X9utaWmT+ehy+Z+yN2J7Qbccd1xaM/sebiZu3w8DZg9YxufO/bcn474zHWCmRv6ztSZu/AxmulX3758/HCvwI0Y4/4m4Jv5aHYzt52ZzdiEjd3JKAj94Gbm3gQAk0/C0XxzMw7Ho9mU3YRs7GxcZ8THY/3q99uPH97f3n/4/EnBnM9HU3s690fj6Zz5/GYzCkebyYzfhOPQnc1nk/Am3Nj2yHXGge2Mx8F0Mh9NNtMJfCN8QPHz/359d6fgcT6b+ZvxbMJuJvMbBjtPR2M40mY2cqZBGHB/wjb8ZjJ1JyEP3RvAbDSaMNt3575tz270q1+/fv7HXY3feB7Y9ngycQEFAufP+cz3Zzb3R3N/47vTzZw5AZ/fuP7ItzmbusyeT0J/ygF5Dmd+9/nT/dfbd/cILYyeQPCqXcH5MCjYYzncFuw45E8gZcNiuxlusn0awJSDCwtvf7v7evsN1q10tuVpdYj4o25qepFtssr2+JF7UertQGj19RXs8eGT97fb+zvv/d1vn8WygCeZ59i4SHzezOvv+QxWXQU8BBFlHsqjgaI4WFxp8LeD9VKiLRh3J1NjQAOPUbXTspynNBux2egDlL1QLMS/MCu0TZz5D1qUalHFCyNmySZgCy20Cs4CY/6zY7tj+t/A1Da6PlhoO2ufo4gbtFJsVvBqX6QwtONPQbTlZQVYEM6oYB58sH1cGQeJdBRqURmlZcVAmQ3QyS90IAWmrAqYqmbC6ViFPXCGCnS7rPRm6sESXfLQoPMl1+6POb8riqwwKvgCUJbnpSzhnieR8lmapWg6aozUAbqUROStAJS2xN1LUGHvgR/L5X2xB+0uec4KVmVFuTR0E/m10AdmTVwWx9kj7Jsuf2VxCfMlFZZtkgwssGdZwI3B4JR2+8o3mqMqs2Kl2aOhLIsFcwZWVGbAyoTVK0sWcs/fRXFgFFlWmQAjZlV04Oq40AmCg2SnCQNgd5nFB8DjjZbDCPW+rlc1w7SclSUHc5Zr/7UUoNAe54CGp1Z4VSYAA1VuiyoKQXHIgmq89FnOSxC3Eo/ESl5ZepsHuTgCEDaJfA+JJSX4wOK9OgCBkgcgbXhDXRZwBBTQSh6CqDBEQ7GLP4GceNkDNcU5Ko52mRVHAEXLUWlIVAiohV/XulUl+VC/Rpdg4f/GBvFKgiiOjT7V8KzHAtTJq/hT1REiPAE4HjAdabV0zR+UEcAdhQSc2VLfV+FcH9Q7ZiWwJo/BIhv15iadRcwJoxT2OC5aqO3TOEofjCQqQQW3DUGI6gJv8JOS5kX2WP5HSQ4KnYLAIFwl2dSPegXQSZeDyK8sdK3YaTyQmYIFaKRwHbUfZGvQcPI/xMTGcCJQkF+05Sl/BKrxpa6fMqNnVAkG4AJEtN7DSf5ekGUNTWAGjwNEp1ziyYCAj0JQdmBrYUrTgcc0iPaXOH2RyS/z9wAhU3j0yjyOqrYbkRrd9TDacqnVgQUQ4iPYewjL3k9sEV29BnmOQIchWtMIouaDk9vyQCo0MpSkP85YUBqN5JBzEdoxgEMTMJhbrnT69NDvlfr6Daq86G92UoNttGNkFq4klLEFVkX2UNfYJhOFIwCpMwvb1OHYHUrUAMCsBVH5zyxKxWSCBPQ2AhRBgTtKJDX73r3jHssOD1pH2kAw0XXrF/nRjcyAKXdPOfcrYEvBMe6ADxm2AJk06AooYBY7SL6QhMvY1vpHlP/a7ACi/NyIco+BzyIo0Lf53hNBefsQ5T5J0L7gAn3QCK46ykoHf1/tgXV4Dl2Gg63zfLn99k0/s6zg/9pjIqA4j8sR2t73eVmG+7g34GdJHvOqu6DPmHPoiSM9ZsUDLzwRAojFrdAVmQ9zGeydU/jiqexDTFVRZB9+L7JQpOzsaeVHvRsHdDf/QZgkXxEcX+RP3mF8HnATQHcCwoL7KMmljEqhmRUBULFr4SQKMHqer18+fPx832Npf10WhmhUvSylH/AdUXXU1z2AL8PId8cy8ksPA7oUDGfICxT4PhSa64PcouzKNS/sdFZhBa+MMIvBWKNzE4pHUrb8BMEY2OQiqzI/iz1kjewjyyjFibqkigs4yqOKFhjDpKtzcuC1nrA0CoF3Ur86JrRtMpLeod59/u3Lx7v7u/feu9tP71HX7nTliTtHgJCcXDOiuGhBIymIAgGus0KB6Zz6Ehgp6fXctn51APQOA7F6gCaGgyEE31LkBfBZLGvC+O+XwC9e2LkJ1XUYO/AU1ZlWtJprE7PBmJc0IL7WfyizLgwTcOy7LvQEKFP6HOLypySGnKDYpxipd/okOtgPjIKw0EpBPU39EcJCCAb7/Q2SZGgQ/wqpARYHrC7MeYa1UtNF6cMrGRrAUo6BPgRoz8SqfRBJEWpBbqZsIhDWowWxS2el6O6a+D+6NPg/IILPkutyH0L2TJYEmmhJVH6MXXIUele6B4m0xNHjBwwbqbX+46qj5BL+2yX55IYJmFp8SJWx12SVqFYT/UoZNIy4ILImy4c7NyAsCLWSEnOsvr9tJVBK3xGMiBQEKNj+m9iTxF6GPgtNv8aZImQGyeiJ05s6eeppVkfTKJ8Cz8ceO3qiSkZyuG1ZxASjqQOBXrbHUS1RJTUOeUZnpJvSrU653ji4FfoWLM6d8XUNAOIlyfHlpXot5sMAk/ThjsXhEKPu4cHRz2MUA2P9ox+/BJZ8QMzZgQdDsWzIkz3kpLyGu8U8YaVvC5bvyAvULNmu9B0rgkfIYbwUIqeIecqXAnggIAbSRHz90+8f3n+41VGctogDqG7RjUlTSouPxoOcI7DhMAss+yNG+00es9Lj+JDkUc7JPoRV6xOxAWHvWUUqLUmdzoE22QZGDhSICUo4c0JUSB/YzMzbZUX0DHFalVUsFpNGUwJ7EBQ5uGCoeFWBmW1T5YCCeYhQeInE4EV1gg0DgrYZSAvnUo7HbhvVAx6cSgcBcC1JokpBYf50NB+Px5PJjLnueMOmnE9898ad2jfONHQmrm/bm0DvQcuz0sMAV7o36En5tt1jdfbHOP2weqARu03wMtomTK4zdX9fQTDi7dAWgfQ8eAUSlprlDgidZlHJW50FD1H+suKI1I8ysPo1jy7475eN6WV/zlb6jykgHY4fxfEQWulRJoyVyxKcJPcgJUxY5e/oADie7asSEgQ0xB5VNOsRWinESxi27oiSKTnWig1rrAEVQfXGTjIfbHkZUb4YMjCzZC1hYu+8cbZFr94Wao82ry0lUB/MB0k+OBleMH197fSgpLIQ9ZjJOR5VcqXk//zyPu0oMJFFwx33H1Teg6A8ytKle1B+RwZ3TRGhHd29vuBpm+qCLCYg5CXVQlQV4asoEYRd4cAklsoEy6VC4CXi9AopJ1zDFBcGatpJggH00QmDQbFw6sN6udTRLuod7SLJ2afSKXqc+TuChvrFyoq4ooSIqurtgUZiT4ZQUpG2USoFdt3JsBGRFrcoJCKXIMugsuWJ4pYsP3p41fMA0VddC/bPFS5aSy9rK+UZDVwhbE2b7CbNwf3EKH41/edjd72bMZwp7/qo87L2GkDARMap5zfq6OZCTlEnSqq/CZU1/W8QwRai1qOGO4FPt4irgiYAJ/ghgwKelllhiGgPUqOKFRUW1IJu2UNFg0GUIE4ukUd2Ako5XznC17QHKI7A3jS3QmBcNRKeIM5MbRcB1drzLLoZoP2xvHoygih1FC3T3gIU6vJ3WcnTGuBKHme1MO219ssS5g60n7ROL64drGWlEkmgEZrPvMjARYzM+Rj+A0QIg2VzgE7NFJVd7N1cCghoNOvJ1LCOLWbAts7aAq3BwwA8kN0pblCPuqejnegFq/mQcEGArPa0UOVF9cswnvCg9uAn+HirAeo/GUfVcxQ9cn4PKCtAtz2I6SFpazBdjNYt1LB5gpzYmNCK0mYpLlwNHThsez9KPkDBE1YKg2TYZmvJL/bANJx2z1t70PJhPliWquywSDLohD9v6FBBYDEw1bTMNI4r3HdtPonfgelYg9YVATJs5e/WAj74ICeXS8UsDFbLSoyGeziT2nxIxAh7SABtCImEPUXJPkFEBARAxHwaNKe0pTGgkp0AX7NYrMBSrEDPNWmWxBGSfmNo7KKhmLYSY4PXlj3pWF8pjCf67uHNGQ8uq/2rV6++8RhsFjBZI5uvSQJrdV6gkfnFgjDaf4hcNHARKd7wa9WOa0gpvDjkLKEb/47mNwyrVb7V9ZKuh1FBvEDPCEtKzgp/1zkOUdakvTCWWkL8GFaq/ol+60dW76JzqyVZz5jOFeG1QPDrDj2vZMVKlGW5LAHWOSwllOibS1mcwg5RSWj3PCWxbCFZZXKubj3/fDnrfP2sA+g0IaUIihSEBbXDulwCETdpeeQ/xFxcpp0Us+ncpiaPS8W6HLX8WcYrJfq5Z6zt0gTZEpmWjFVkYEiKjAmsJxESUBR2TQlHzkdnQfcNtM2gTq1EW7g0St9TE2yKuGAQSHTGZmKI8O4MOPOLxltu2TLe7VG5y6Vh2qltWGEQLQA+qxF7eTDmx2Bh5EZCnsnQY1RgpIPXrm1CihgvHT6cmQV+2ReBtbxDh75mC/q/xYSwRhx8b1WvcxYOINY0Zwt3ul7XmE0FZvhV+9xGWbo1zZfxlmLWLO7AkwT/MwCVvCJRSxowWpBOXVJnP9Dkzl61+HafYMip5uaICC9qIyB/rYCLRwr/5gZCTT/7pkGmbu1imKxwYXEU19XhZn+bVhJ9UmTtxOIUwQokxOuYaLtTfrZlRi6UYM/YEIIChrU8A+VCwfYSlJe5jJi29MZIh87gZ8e+dk7dPWiUbf0ITMS7BdM107PArH5x6UwCqW5MEc2BqiwJxyTIIFOWdOjWdadzNWk1y5ldNvIvFLMvWnktZ5juLb8/LJ6x3FPnos8W1XpFTbmseC452dBk6EwasrQiYXrp873lcmAtlopOjbwpXYYHfnjLK+xQuW2Ot7LZvt3l7wt8IyHr5/TuQdXOdU+/TrKAfl+uo8MsWejZbgC28ND6+qTG3tpf7dutr2NNXdCOlAfR+QHJEitWPeKgfYXfwV8A0HaZAkojmD8GqccCckIA0X8wVtJUrqpFde1MBZHoJoA2Wg/+zDYnjF2b5G9WBGsIecibPwmp5ksbkPRyMqgHkbWtNzLfFAXcpd1kOh3x6D1k6SmXIcRM3bwMXtAoYQWWz8oenJpl0d+KRGoLIRJacyRMhZzXTtD3oGjzzrWzkFpTk5JrtETXkLRFgwiZWy3JQlCws3c5/UMMqClRK9vaFJiu4LNxdQGP5Ql+hLKti6yXiMsPSFiltm80dVjsbUTrjSaPit21nPTZwA8NadXapkcuu1ge6QKS1ZWFuE0am6cxIT9cjhbF3pfGJSbteLJ9z6Mwr8VJYd6VL3RqA1OVTE4iBQXmF4gy2xULCazfraZTReTt0jm3pjfWDqEIIfWGtIcKOhmYDz9BFIYwTUSvvyyHGAkPZOmKOq0kSg05Mu0OsCdj8HaJSrUC43KNE+rN2vbgeilxaaPXyt66KP639luUgtKJZyz4Zqi+jsP4qtDowgMjR3y1HHPx1GkHbbpA1vClkZal8dHqgf1WZQW93AcVq0rxWoaCrdfiqgjLh5C753sslSb4dCpKkn3FNjHvgiI06GqdH6gAdoma9kBcaUIPufUt5tdq3MTi6hLTO9hAH6w7W6DViUyQJzwcp39OgM+aScC65GqexoSmSLZRpPGtA1qqWoaiNUIbWo4tfu0JWCslRGrQnuBv32Kpv4Jv9lFcLc+VTyRJOhiAt+H40nlJKm2IOhD+sE1pALpDCXDQepDV/pMOZomr5LdJAJsklZpvMTWa0I17wHNOD1i1O0QSS7zgXKti77cudiz9nHGn968n5r22G23TrozFX7fsUjsa4067Sz/tdu8NElMsNb/Xl/CL9oW8qav4dSHyeAxzzF4AvGjMU/O04jSWXjRektIfiecmq3ZyCoBqq7apRw3RxXQPeOUBiwFLybTWY44XroYXzlwFq2V9/4IuUO7H4mibJsiAhXgN9Yeq5ol3hOqlYslLvIWUjyIEi2Tfkuo/sgFSFAC7lrL5un4ZgS9VlzjUeq1Lzb/0Wrf/lpEeyOK7XXwdUki86kkfvnjv7379eHt/975fI1LymlN0KNRO4m4V2zjbGPrP+uCMdcBXUPj+na5XZJkGX8fTu2TxOl6G5ZiKU2huAUIQlssXMJQ7XjAKz+JRsJGb+C8jcqv9yF4RenD5pTDStftQ+MV3wi21wJVX/w+U+5ZE')).decode()
assert hashlib.sha256(COMMON_TEXT.encode()).hexdigest()=='ad428b80b3a28d9a427846d06b494f31d2b8264a5fc10084a7f09065294463eb'
COMMON_PATH=WORKSPACE/'common.py';COMMON_PATH.write_text(COMMON_TEXT)
shutil.copyfile(COMMON_PATH,SESSION/'common.py')
D200_CONTRACT_TEXT=zlib.decompress(base64.b64decode('eNqtWutT20gS/85fMZf7ILti/EI2tnN8YMG76zsWKIfs1Z2PUo01IzxBD68kk7Cp/O/XPQ9pJMvA3h5VIfaou6enn78e8e7du8thv0/SJOSEf93SOBNJTGjMSMopO07i8JlcjvokS3apz2Ex4CmPfZ51yXVCooTxkIQJZSJ+6L579+5IRNskzUkKEpLo6AhYvcvzu/OP8ztvcUnOiMNgZUtFypn3sN15TwNvODxdD09H/fU06DuS43Z5c3dzcXPl/Xz+8WdksknWJ5Mho6NTNmGT6WgyGQR8dDIar93heDoaB+6IjQfBesoHU/gw9flkNPKdQhHvx8X1T/Pl7XJxfYeST91xMOG0DzI4G08nw+l03Z+sT7nL+67Lp/CPD6f+espOpv44GE0YqHJyygNY9Nd0oiT/8Glxdel9/PkcRZ4EjFKXuWM3oCM6GI+nbD2ZjPnkZMrZad8/Hbh8hMc5PfHd0ZgNXFDSHbBTOnX77noNIsEl3sfbq8Wdt5xfnd8tfp2jYD+JA/GQ9fKUihgM3mNIlz6sPf7E49yT694TDbufsyR2jpSE25urxcW/kD/mWc7ZMfr6eJvyjKdPIOW4cPvx08A5OjpiPAA//7YDH3lbEcfgqachb2U8z4E8a8+OCPzQDPhzYlZXDuNPwufOPTlDVbc7R0ZR+ZxHu5DmSeplnDNF5g6bRYFuSfgE+/pJFIncyKT++GTiuu5odEqHQ3dNx5yP/OF0OO5PB+NgMBr6/f6aOc1Ct0nm5RsQrcSVD2L+YD/oVpSiYVicfPUoCfokSFLySERMVk4mHiKq2Tug4y5PgsDb/I5fQk4fvZTmXH6XUvHHyTZJ7sWJyHj5lMChg5T6YKBnb8tTkTAPFGpLrpTnuzQuVNY+kn5jYKVsG4q8BYnVAZKQ+zn4MutADn7xBMusRW9Dsw03HoR0/QfnW5KETOZ/Jh2mpBJO/Q2J+ZeSmUB4xf6GE5qTSb837Mt8t0wV8ril92yjnfA7qFysyUWIWMkD24BYziAuMygYnBVk6sSyfnSX8r/WsI8uHp62u9lmFwQhbxl2RY3+yMTvXLoEzTDowy/Y6n5WWF3kPMLNCgOtkOO+eK4PgWTg1cJiZbAaJZo5GI8SVF/RGvVWM7kJePdjYUYmAqihGQnSJCI52DNM/EcwxHWS83WSPPYHBDftJSnjaVcFs8zsDoHUhiOEIpPeXjkq4XFrDJWO9QQoBaPyBOZx3VWSueIotSKX3L4MBnwCoipU+F0uDPq1RNMCuiJjIvuciFgRN1KR98QIxjXLLaP+vW3Odj0bGXpZyUC/y693y/PFtffT+d3cu5z/cvNRB5EM6jPyjc2IspVTssjv30u67m4LBuMtSVyaz+KAxe9acPwAYg/EKCF/JTIjZaG18gd76QfIqSeekm0SCv+ZmEpunIz5hRqv7ou4XnP4zTuEBjnwYYC3dISDw1sm0NtWpFPGVF6VNpXMFbOulNyZetL1k+1zq12IgAMWqSbFtavSPRWJOnPl0mqGAaKIe73RfZ1DeazGU2GZWTyF0S3mmXLUit1jM1P+PMQA+lXJLY8WPMbgXbrd8pi1vll5j9npzArrW0/yBNal2cqSLsu6papOupmtfsdQ7OXmrFRaBxjEObqp6kSsZ7YL7dbwzclp9giSnCsR5NhOFGBTvQHXEVPU11V1mxWlze5R8nnE802CFDaUQCFgtojC+sBiqZ5c+5lZntGHEoHlmcKTbUtSg4XeLs7ytC0z3kU13wzGkDtyvWE/t2/xbqCuJqnwaeghfNW229DhaFzYBsDfYY7qxiYcEN0kcdPusGT7ooxKEfJy43pfB5EFlvNMcKv95MeKSMQNkPUcY8P4v5TYkyb9oOuUO+y5Jz3X/UACEUPmc9BvJzWG+tM/HvTdqfNdoxKobiJ49hQ0RSEtaa4aasSlVTXIpPcqkFXiR025F7X7PbnSJDVfpUN2zOp+d/xDzREC5/Xu6P6P3VGr+EIA3bfJ384siXXdtYRD0XV/oHm/zqZ3LbQ2sBQZtfdhrNhSmBuk+xUu3abJZwgqC49ud2tAKpuzu3THdVxoInDcLc03hkdtA+eHdR1XFZm9hpFo9MJEpOTZGFklDmKE7x3492Y0uQUtgWtPD6le5vRaDnPeZ3naQkHt9469f1UHBUNBFlJ0cZTOWii9i/O3l/Oveatd8il9VxXpyAw1QRYHydquI+z9nHzz7KAnBiWyduqGMVWSQUWWJxCZ0qk9q2Nmn8ZJjMHdeunUMlBL0gLzd8h/qp33PIbTQsJAAJDyOgP3Nmi7SxaA1DNUPZdCAIgl4GXIl5wkYJ8vKeB4pIC2knzhTANvHmZ8T3sdv1Bul7u4xO1D2DHN8pkE9SlMwxzk1PWBHeIkJ3L8jvNuCUZgNI6E76E9pCU6pDrj6GQzi526y1X+YfhjsU/SIoeKYW+ppmqJQ6s6QqJTyOyOBqeURLucrkFbGNh5lh/jBdAWKxYMJGboeylj0yTBBxkNuOdvRMjKIuDgXj346nMwJutZV0Gqxvec99VLo7bZLk/8JKwmSgt36jnmoc6y/dQRMZYDGK0b2XVzKYgOitmLXrOxjNS9myubx1CuHEwt3cJlD1ue/1PeHGFHKakqGMNqjUDZLHa9AztXGCoXU81MsjAILAO6v9r73XxaXswbVANo/DsHzMHDLWRWhefH5c2/59cv7Fd2lfUuZmF1x1/Prxbg98XNdZNBlIZQaHO8JlEcFzfXMPpd3Nm7FW4EnpzmOz2SOxc3v9xeze/ml+piyiLT4Q8FrzRcGX+HZCt9jIISgpXs+3GQch8memw19s4W+pAe/782uj3zq2FAXX/J3QyK0crJMB4pZLB+lo0fW2NaTB3OPUxVskHKUVSzfa8jEM2rxZl7wMqyhTgq2Ob9QYhWXvXgQgc3Rx200C5ew2QNrQbISg9jHVbnb5kLAWhW8pwibsSMsgNUhwq7HGMB6RQW7hhtdDXWMZLu4lxEGLsR1GqxFrDNcysJoYj7uzQF/2m1H/mznP4P5SWUz+bkw+vDsqyUU16RLRb0N9pAcviPSLt9BvwdexH9nKReJOIkxdWHlG43wpe3mvGTYIJ6oVinNH22tlHjgZWUcptIZBHNzVXGY3ldKs8HBoejdx8gAB7b5C9nxgZ6pTbW6hSeqfyF4vDD1Vw6DbuotZFy03L+0/x6vsR7oMW1d/FpuZxfAzj5dH23+GVuG6HgZNBDechwh1LavgG9HTRVGj/IcRnUP1CP9FEantqzlxm7BANa7CPm7YqItiGPYFFGWgcCE+8leOw/E+ktmUnGMR/ke5nyhQx8jKhMSv3ephjJ9HW6jEwdUS0VuCqBywA+u05iXiIGiaug6ceWigVCUjgCqjggBkQT/CscVEtE3APKwQDzIOICMmhYIPECfm4XaNE0d5COGsz+ZLvXhUvpYhWvCi6WxaC8SbILwV7FfLUo2BXwjVJww5oupTL1DZrm1OJ4e8WtIlayKE9YNtEBqta1XMxM+X3lPIpYN0NHTg0PPOapOupehTU8WmS9myoB9iu/oee8b8BQq9lgeP+K8IZ2W0jyajSl5INC90DWIQBm3TOGGPAVcKtyyQgtTeWlHAC0eOKmeTVOEw0mN8Bcjv5Fhjt7c0ihrKLbZRyDbA/16Kr5B/1We3WLped1ZzRjnzeZvopvZdCGTJs2ZGWLxdUCS1nzDmz79/kFVPubm7s9j7/WiUvxZTi+bkilCqhR6rTCRN2PYvn8banaYKzGkEJuI7eRoAEJydSPaCwCGOsq1jeS9h4ekgJpx7A4cWig0BfTbQrzYVXWAZIXc6n0+evppF3g6dHUU/I6xAOR0MbxrZp51rJjSXURVVF7+B/Ca2y9OVctpVML/gIPOO0Gz9b2RwuoT5UbNjOtq0e9wtC6iSHXQQcpVKVXrRFBtXJzbAWLX85RfQWqJZ81Z6M6pLSgqakM4WWafMmK3Yw6lVHDLL4YHq9EUP19XykTzgLDDsVSCGWuAKj3+r38kwKanadq8y3pzIzQBn/e8vRYNmrzmCg0lj8XV0aOjUTrx9bQ6g3NZ2aLaWh7WpKIodyE+3KqE4a5LlUXcSWSqoOiA/dwf+r+rdpOZS9ruLWqnyuij9wcSnm+pec3KHilPTuFhe30R4QKXY1VjfitQHQzHfTVwjrTsmvDgxr+5V9aoEwkkx9saF5BRbNCCxi7aMQz/MMMz4dPKb77KgNz/6mN99WE/gXaO+RPha/6BP9MIKNYh6pUGjrpWctQ2Ds0Z9Ls9XSEHevFZvZqqbJnSRw6fajQ3tcoLCUcStlGcvsgKY47+BdCMDixqhVqj1D1MMHEgwR+zuADHI2KcJdK6/W/H/0X3y28PA==')).decode()
assert hashlib.sha256(D200_CONTRACT_TEXT.encode()).hexdigest()=='2ddcdcceb8ffbc7effb869f4122737d36548859ecb5005aa71d38dcfaa2222a0'
D200_CONTRACT_PATH=WORKSPACE/'d200_contract.py';D200_CONTRACT_PATH.write_text(D200_CONTRACT_TEXT)
shutil.copyfile(D200_CONTRACT_PATH,SESSION/'d200_contract.py')
SOURCE_HELPERS_TEXT=zlib.decompress(base64.b64decode('eNq9W21v2zgS/p5foeuioNyqcpLdLvYc6AA3cbq+TZPAcVrseQ1BluhYG1nSiZKTtJf/fjN8ESlZTtreS4DWMknNkDPDmWeG9IsXLyZVag2TxGJZVYTUuqEpLYIyzlLHiqoiWCTUymnxJqLrzAqz9ToumRWkkZWl1FoWlK3eFFVaxmsYVtBlEt+sSvfFixd78TrPitISH0m8cNe0DKKgDByLxTdpkMBntciLLKSMwfMD/FfSdb6MEwpPRRDSRRDe7u1FdImkGS021L/Lilta+MsgTqqC2llV5lXpWGugEdzQ3mDPgj/R6l0G5UqO6PH2MihuaOldja6uxhfnfYJUaKRo0k0c0TSkpC9fctNgTV8Tn7yuqjhy8b+f7J67ovezwS9zk6S7vo3iws6DgqYl86ZFRR16H7PSz2690yBhVIwOymwdh/6fLEtt8aaYBCzFT7KQi93FXuJ8IUIhPhCmYZkVD2TAykKtxyG0KLJCtKnVPwouy6ywcitOpRzcmyRb2OQVkdLBvxhE6sbMR2HbPa5PO3dZtVzG9/jmjMhpEDdkG/zI0xv8SLIbMu8N2Koq48QNs/yBk8gduZycC60ntAaG4UsF20LIjlXQf1aUgcpounEspT+0H3j2ft7f363DI7merxS25OR9efVKMSWCQkOSj0emVkRjn8g3fMFFCkM2CvIMVlzSyMOpu7BDln6YwUaghd07SgLgKwfwwXdxubKynGoGrIzgkcvTIXcEdMAs+KJVJAXn6U3iXnIKM9gqLr2nYVXi7nRwLUK6PUevq3sJvblTM8A/UIKHihCz8WAC+AimZbK9mp5cXE8dvh4/pXc+g1awVC76Xk2vLB4GDeJ3qxidh6SSZ0kCphYz6xxcR3Okel2NvQvi0lY2sY5T+62zDu5td//Aka1v7A65v5Ei78HfFn16H9K8NHyOOxWkRvc5bLFokAeMbb+VBDnr1vIbU8HmH26uzkXzfSYp/s2TKxlYRRAzaqnZ4La2ySeuUGtRRbCr+NxpBHO0yGup1hmJIzLvfSPzrmWgsf7N+3F/WyXCDOO0tBtMHQIbO43RIRRAJbLlksApMeIsk4qthGmIfdDBc6+llHcBoyP+CGaFOwHam9PZuaztWWfMvY2TJL+x6zfiyBExx70av5+OJh96X29+B/vfYUvPzeG38dlZ7+ir+D0T+mAqzfHcmqQDZFVSerjxwc8EEbNr3yBpiRHSNbgFDSK/pPel3euhwJ8eraMHBZdrfXnca+mpoGVVpGEW0b94+xbEJPk+GLQN7i8oKwZuL81KEXAuh1dX4ArPLoYnw3dnI3g8vvhweTaajk7IXGtZxjpP7RARw4+ABsshTsIGMX2g4WV7r4n1L9ltzkTEUcNhPCNwFWz3GgK3hgxeQvMVG7gxSrKrcoA/io6O654Z1iGQlT7KzNsW4zaGUMsULr6hIEd8URNAGnIeIjavgiK6g9fAmScwD5i4LSMvYLUigGnx0HtxOQWodDUjKaCjOPBlJ3ieo4CvWA3X9uBY5Pzj+GQ8VF1WlQYbECJGK9e64vysQIBHa/qTJfGjS/ZUSILtAx9xAYaLEAMiKnxV8vtw/feL4wv//ZlH6E1CnMvfLy5H5+/P/Muz4fT0YvJBtstxI+g5GX0cH4/88YlH9okGSbfC8s7G72DM8OzT8Pcr/+ridPppOEH7+zC6Gvpoj6OJfzIZf4SPC/hvMj7B3vfDs7Px9QfZAyaKU8yz3L510C/11GKA/okPLCbDye/+5XD6K5l7ZAD4KoO4FsVh6S6LbH1LHxhEdrAEJU+hUxHSVRtILEs2IGPVO389yzXewxlwm25zdAhsWJYn4GaANd/c+VyaPHoGz0Rq6s1PF5PfYMlIANAoOP4BwbFArAzYrf6mTGJgzLT36CCsqJF2y3qcQ+nkNjSNskIjcog2G9CxL8eLbmHQDZgmOmBaaHI+CGAdlDC6QFgCMztw9919mNr4+IQMvqgZAhIqV2SA054RKcpINM4fHw2F+dxoPo7OTy4mtSBPx2ej8yHYBOgPVyqmICwT1d791sl4Al5NG8QCIp0H0y6yzzT1BcQXkjudXPxjdM7FDYFUDVgFh29/JgPZefXrUOM3InYuLKHWQOfSnK2BNVkxXn6bSw+ewsq4XzKNYgLLGk3AHBpW8eoVrscRxlG/qC1EN3Xbg/HKX/f1vozQmOs+YdIRpGcwE+LM5r3Z4NCICHF07yFGgZXEaUTv0TeVRRwkiGyV/+ANM9LlE6Q+gcwTWBbksKDfIBExXR8FA4SlhDgRLR3x9dHhc6tlYyvh8G5fBCycXK+WkQFBmmGnOecwA7+aVnSvCaIWVEj0pgjyVRyCSL889kRTHRTq/VfEy5hGhMMtxHRNDjKuiAXgOtGgHM5jb2doJOeZJQOE4senCsDDtY4vryGkJwmm/shzAWnxLY1cIlNKmRRLPGpjVQJyygxLBsC2zMIs8VcBWzmW0ICjgpAMbnLG/6UdCD4YkiXACAIqCDJdhgH0WmNrwvApAmf3e5qZUoacc4MnRKDx6Xh0AuHregJW3WTdflOxloNbXkWMaUiTDJrCJTyWaDqXZ+MpJ0NQIWTA1UKw2MMowBJQDxlwJWk2kM/5yyIAnEQGNcjQjT4kDH4IT0WAHszYTlKvT7pB8fBNnlC9op2haW+6KmYXWVaidWnDM0sa0sxyCO6Q6Xg4uM+FwkgfP/okzNZ5QksqoloNmsULGkXpjbbMEjAcwHOhF64o7gZfjhZzkV+ck+F0eDWagkdzpBKJdhfrII2XWA3htvAgLcGWtPlq6hyv0wLAqXLuyoaiIrgDzd4xRUNx0DwlQISJz5q2MPc8/OQZKe8N4SlGcAeLT29owZNOHKaI7hyy18xTyYRWDPotZfJSgQMi1mgkp3stJ9ZahoOD+JgfrKHMMvqAukpwdUaRFH1USoEbujAW4w6PLKXiI+i6A8COZc2SgbfbUN4iWDGXk9e0uqxF90KTjevnUDnN7njC+BmAhVuVIaC7slhii01e/v5y/TKavvz15YeXV//AvOfJCqYU3HvJCLN6YQ5o2215mQHwcnJxcn2MO7cdAutym9iv6AtkyMuWyyROqYQDRq25dgHSvGQ27DNV42hE5B3WbND7jyzaoNO06qfKFbvSRoMY5upHOj+X+0PbOMwPME+ANej5TMfneZ19RQCPweXDxji+nkC8mPrvJ8PLX8fHV1tDpOl+5AKCCQaQczEKvMo4tIKiCB4guFJAW9TKq0USsxVum3JFa1dknVZJYo0QheEuBQssiypURn+elXSRZbf7PxJmIe7KMb7BUC5TYdryCGD1Nn/QVVD85p6ijwMD+vvoeOpPLi6mfe4h+qCCfhIvSz9f+fDsrqLlWwBMhaiPLrUF3BRZlXtL4Vj65DXqWzuDTZAIX5EHWI4xdcBfnPH6A0XRDuaqKQjFNtNtLmzdgkFYguQ74X4ZYsgSpOLTDa89N6rNIeB6WdYYGLULh9yCeKBJ+mQdF3F3mE4b1+DrLfNk+CxoAuvZbB0P6JW69ZAy44ECACRRtraNPvSL/XqULPUYbLu98OBZN+2QsKABBl5wVmQA/9m9x61yhopksKJG3aJjPzpc9CJG80KCX58+qYDUFZezFN2P9x1ujMgqBYCSigWJZqfxvCBODHFJd9dWkphgO0fezeBZ5yg9iZjADMtBcfmAfkPaIzgMUV+TJ3dy2OqBgX/x0yzlcl9C8JK+p/3e01wEtPX5nvK5q1HQDghsD+9AeJx+mFd+d/VH9vy3KkBaQ52lH48cEKdZ3vFIkmzWeZwrTLWrcAEL+x+VLfArrgQ5uCyDgapmIaXzjXULJdOvr13gG/Wh2u1gI2pozgazddnuxoB1mM0rTLe6trsF0XcjcVmFAF5fuUuNWTXyhP32voVx0uKNFFxuAegETFKbhZRNx940iRz8IvPxwEtzXmG3n3qhv4zvEbtLlvyE201zEAbku9mdn8fhbULNmLJ4gm7LSXw7cbmbwZZWQU49b6Ee7J+cHw/FmXCKh8TgyEGrdtBzA37w0u5YyA5OFTogriKMdAUDH/rCJGNAwFk4sBkS74C++atT4JNR87mlD8JWEDzzY3kWUvSmBGEZf/bv14kyE8yn78ChlJiib3ci75IHNlS2cEwc9GAfj/Pyq1FHSuiyeVjypDI1Vus6PtHJBSrnGaptVX4laVV/h2nPQHqYVOHr/NmxyHFVFBy6ydsZx5fX/feX1xaXlrWOGfiecIUnivCCjLYI10wUI89kwJpiPO/zRXYvgMoBeioOhqD7TxHefOnkB22nv1Uz8em6SnhI3wo/g2fjkwlIQCU38K9p/lhHCBawgiUIvbTBKPH4GD6g0Q7eLHo9E9QwiDLAljRvs8j7MPzUdFVQanEWvFIFWMySaBFSvAwQcxKHD5YSUx/6E4jPZBvhGPFC3ZVRpzUoezwS5NnYacdU+nJ1+p6NJfSzlaSp0x4kKSCSRFCARN7uq9oCXguBXNZAoOoRAbnTzjDEpYdqvcazIcNGJtfn5+Pz92KbFU2IZ5gOHgBDvKPgkrkTcoiQmo/bQrc2tALQ3ad5zAB+M+7Q6X0ugobIkQdvoU0l23XjDNEmvKID6/HFyQiC2wRFL/sU/n13PT47wdLVo0LxWRExbzY/eup+h8ztPIyN29mpTEplEoImvqjSCDD2x+HZGGSMQezd9fnJ2WhrM9dw/Gtzo57nTYafcAE1KV6bqzNjfujTpAYmvIxvWL+2VzAKv7hZiKzG5804e+l99ozLIqLIKyK2qPN6XQeJR23j8fhYM6fV0y2D8Nb7kg+2b4q5UoN23tOHXJDpVuucHzJkRbhSn5uY6xpSnOpPyPExQGSLjFUQoRSmMFLScHPYnMCM4N2acAMgCfpcXxmP75sXcnj1AIx/h2UtqjiJtk3r2wqw3Phl8XlFE7A81lWEJlvmVY8yzIyP5AFU9kljabHbWc7lHZC04IeqKvrcBYL/w+WfTyfD46kD0hPJs6rlthioWMdljUkQfIAHeChXMP118GdW+Os4xftsScxKG284KQXE6TLDA59ei2RtSgPAolvlkNu5ecIrMC/RRXji1Dp02pWTxxaj5tnl7ooxyCdc0XVABgctCtLTabl9wVIPuPqMJ0HZAgtHXJXw7i8gTN3gr7Ii/owdhw4GDTwwbjQf/OxsXU8h4nZYY9wv6Ph46PJXn4Hafq1O8RW+L+OClb4axUqaQ0dbGrAx8xKTPRk0wSXD4uJSRL++tkprTdcLkPEqzmWhd51Br5UtrRXoGOYVwjdeLeJ5/5s6Sbb4bT4M9aL7SFdzhf5UJOzXIMKCIIdoFG3QqtJwFaQ3EFTamsSEK4k/S1ETzE+WiCEjawVWcWQUtbCdWeejT3VUfwOw5AGdRQmTQ7vTTkUJMoAwHAYIV2BptvIXvSMdYj2CvlZUpThu2RzgCZ9+fzY4ONS+0QzIXkdIkNkRr0zXJGWdqa+5dtJ78hpls0KOd4HMVxG+iLVtXQYayECmxWBi36eJmCgX4pohFR2CACMMTED1FEWn1oDWlMAvqpCgK3GeAYN04S0recpsMuk5XcVkc7KO8kxbRVo9D77VvBpIH6m0unW8yYdJEEUck4cRh3vtk6K6RlbQCu8ubp9hNYBek4lKpbFm1hbbTMNWiJM7ym6dBCAJFPDMm3F289ezSB/1swyRls2jzExMRIK5+WvZaEQ72cNrDNFfPEHPkEHBD+eQbs21dWqdQSYs18uJyOFd/MXlNoM9aV9LxIFyFoPvU4A+buk63TY7tlTflLKuQz9fSu4uVYsJwxD+2RFZtmt3RvG38+hWgMq7GCLwHTNHNzvmHcxYgNC+8ZL0czKiqgGI+L+3Mr3NdneRXOh3u0BuHNP5quxulnobtfiGN+l18O8seAx2HRN1ju5alqrS4lFO1BBqqwekiT9GSPwbPCWCB6oOszALe2ybGyZMbpDj+Y+Nlfuj2lu0U7O5NytmtW0KhFaI6iGnMm+Q7syblSVLqC3duXD38lvvCAtDAFPZRr+3tQXRf98UeMGS/8BBzqB99RaT8YSmtup2SP/tfv27mH/Jo9LaUB1+mq2tsuu4uS7baKqeB0QRI8HK7N3Sw5iIA77NTe6KfWL/e+b2Vweh3FHLyTmCj1yTB0Tswljilg7RJRbb5/3S3bbM3Zjuf8zAcNEtLtItLbJyJf0S8w5fSUYdPmuLpVl024k9pPD0tbX6dzxdjnbXGWD37RvT/cv7AQM5t0c9tx+sKSBrfhh7b4lsnFlBfRshskS66fC6fKCBLeKuB0vOyDVBMDCEgYwCSASTAdSSAmTn9N1OiXTXGWqUKqEpM9Eq4uEEz2LL7xJYK81Ubhahm3DADchrumFzrng0Wot8y+s/iVpbM6jV/xyZbnPpGUnk4/N3DjTqkQUpaDzasc+H7y4mKErcPzGLU2iHCGLDG85v9GGRBUU0xvJWUeWlvNNPTofjM5Q+vyPPoTBeYXDqH+d59ZMrD7OgX9WoubUkRl2sNTEMOrxY6IlaofwViQpDT/3kpns//v+DxHMs1xGkNYIRz2vID3p3HVpiF1i8KM85/5H+kZLXPGGKqnXO7PYpm6LbfcpmHkmbrlxU8IWesLwR84sa3qHh1kQFyVM/gVQFS7k8R8EuSVUsUoxxP8c56bWvYV1xoxvU4tYnyg75I73OMR+EXkECm45rJyUAO3G2wu52oRuWLw2fX8MCOfBfBIlbp7LHvON0DjoYmP5N/fYVdtTx5TW6Povf9eCOb0FpKpwk3j/dVWWXC9z7NxTV0kE=')).decode()
assert hashlib.sha256(SOURCE_HELPERS_TEXT.encode()).hexdigest()=='79723818944b1ec45f6711362b691bb4360706241c3330f417f7d04753c0b83b'
SOURCE_HELPERS_PATH=WORKSPACE/'source_helpers.py';SOURCE_HELPERS_PATH.write_text(SOURCE_HELPERS_TEXT)
shutil.copyfile(SOURCE_HELPERS_PATH,SESSION/'source_helpers.py')
D200_SOURCE_CONTROLLER_TEXT=zlib.decompress(base64.b64decode('eNqtWm1z2zYS/p5fgUsnA6pmKNtt2hv6dDOOrSS+c22PbLdzp2o4tAhJrPl2fJHtpvnvt4sXEgCp2DfXfmhkAli87O6zzy7w+vXr08P9fVKUrAjLsI7z7IhU7D8Ny+o4TMiqZNWGlCyLWMlK8pCX96ysXPiygg/ZklUkzCIS1nkaL8kyT9O4rrzXr1+/evUqYisSNlFcB2X4EMAsTlHmv7FlPfJfEfgvTou8rMnmXfHE/y7CekMm5Ar+aXuOaRTW4RgEjJN4VQfFBoV5m2j1jvJBYVUxEFJtwmAVJ8xBISMymZDZ8S/B9adjl9CzNG3q8C5hBIYSlFexmiw3YbZmkSfElPlDBXPPF/yvhxhWguvyPiiZIKekI5iOrMTq+Q4iHFTBLljkrOZ8rXTh3bOnyhmN2m5yjQnLHBjBVwen0Tav8pJELM1JnKHETjz+ty7zpoBJpPQx3cOuC6NPVYc1w5XwznMq/qaLub84IuESlaq1yg+82RCTQR9coxg+MtrkFjJcO/aRQvhe4qx2uGwvrOuymtOsSYMqTIsEFzHiBpKRv0/IwV+HZIrpPNBgwVCck7nfHYpRchaj7cfhhRVeXK3iLK6ZWr8XJokjZ9da1cpFs3kAhQcD6zhbe0JsAH2WSV4pmXPf3V+42A98BazHyUbjw30XzT+ZHLC3P4Jj4M/9F8uFpiTOwmTtZXmZdtMcfOcf/Lhww8e4mhyMXHLQTfLD8CSrMky5ETjZ24PRtwf7ewdDByW7/W1CLq9uzi4vrud0E5ZRkIaPgWgLClYGS/hVgi2D2Z+yZRKWLCLw/S23U96P3DXRGvyIPS4Zi1h0RDK2BYioyyZbwi6kY6n/0MG8sCgASZzPFMUEcUR9bvgwiTRZn2T4h7Qd+OvtAZwp7S/Mlxv5Ig7hoQTVBstq61xPr69hW2MqMSfgCORBE3X5IsSAktVNyd1NfJVwVeVNuWRBiQBY1Y5YXJknzAWEzOt8mSfBJqwADSK2jZfwOYnvyrB8cskmruq8jJdhMvkQJhVTILfSWjrXrooEcZElALlbMG1Cl3m2itfVuC5DMNRsPY7e7Qfl+i6AU83qgH8OtmHi/VblWXe2AGVoWJUhiyMFLBjguWLRmEuoJHTs0XEaZvEK9meL4msCXwMR11fnZzeIn0cA6g2MRnD8ft892BeYwWCHX9kNxpRAyJhNz49vzn6eHhnytQ7WJAc/7Lvf7y/6G5Th4fT45vh6CnIvL2/GVLXSMe5tTHdtTXaD8QPGtz1k8MfV2cXF9DQA2TdnFx+vXcN8abXMC+xFr+IsQ2eAgCkOGOR28vMseQJPyGsSYpdtnDdV8kTWLAOrhShBpqgLFYQ8+qWLJStzq4BYIqKNfOW7yzDLM7QjB3fnJXkYVY45qGRhFNTssYb4w8G5G6M6drDBlShjd4AiTWEu6Q8ZMreqLnurEI1BnTtXs8t/TE+EvkaG830Gv85/Z1kgvA4OFyV9mF3+e3oRXB3ffBpx1xddNuHhux+gi2zGyN4uiip+Egh+IiXNphen09l0FvxyOfsn/KNEWr072S2JGB7ZzQc4F69iFsmVm9P+PJ2dfThDS7q8nZ1M7dntse3ssru5M9nJAB/oa4ERbV1LbkT9qUsyfLTtpD6AEAEx3BOhWXjkHOMdBdiJI04O7eaDhT6FbRw4i/1tqH9fA7rVjHsydF2gD3euLB0rQMyGrxy6u75dkIO2k9sZ6Pkm+DA7/mkavL89/Ti9QQkc2BU+cJDXV5wAKwXlSdiHPvLXHMyqypMttKEL8Mhp9+722Q6SXxZfzAAk8QIPPELiXOZ57RI9HMkQ5RKWbbvg5JI8ieCYlnnJIxtr2kBU5MDTID5PCAob83NrYROIO0RcCNodbAIeySEaErUnAVOAJA2GVF8dgGwCDIMEOW2AFyI40ZPLn67OpzfTU8p5Gu9wH2eRbMa9Kz+5g89gBHRQqFJ7LEeqIHF22slVsM87cB7TNelWw9vxl0bQEwAMl6jggmcolB1woWKFUkkgsNNHdwRFiTSZzkAjGDAUCvBASOTefNrqmJ+U2DhQh3WcAV2XJ+OSVdJUm8lN2bCRphCOqfZSh0W5BAcrNXMr4YehshDNiDqVw0fYuNY0NzMRqQ/o8OyBinXwYMz35CMob2P2AL7CaY/KLqnbQqCmYp+cQqdOx3aoHsZMHAOwcnN5cnkefDq+/tTHzZaQWBI1d9Sgje90qGnxxfKTAc4xBFTE0pMvD8pajdJtt26+EvszIhBE/wiRm4EDQ75ScitUA4YbtcX/qXYvNGrYPmLFW2RMqBsJfZ0P2FZuUk6xtmE063QCnxzcYR2nDDKsBwd//J5nzGvq5ciDcL3CLw5986836Zvo5s2nNz+9uf43He3RgO41TRx5+L/vnZG3YY9z/6+L3s4+yslgY9yX7W0IzQ66bJMFkkNivDu9PcGETOcLLvn87bcK6SmaD5dJwXTuqU/z1QqSR0a/yCig9NVmdtKccIt5A4ZBteXX5ZNZadC0zDX0pPSrxIr9dISyVbZLdD+amNzezFKXG7a8V5AeYY6G2ZdjW5rGUSG9LGryHpxlyn+CVrEKA9/N5QPbBvzZKjIWrMI4acpu9TBgdESA4lRfxSEecjoD2glA/yP4DJ6aDT/mwb0Ig5Bwih0apBudYjQSnvwNOcFD58Wv/AGEVJu4IJgTQrZClmVeVW9VUUyEgju2yktGiuYuiaEzWHaoQpQ3WLmz6mUGeftKAW+wpvZcwUtyUTjiMC6x7Cb12699uUMVL/lRlarSPGIJZzd0YR74qkkSkXtXIqG3rFzYztzUofQvGQqxwqQsek5B8SAtxKgGsXxdhsUmXuKi7nmA3PI64L27xQCsjETv58U1S7GuCMd2pUoxqpnEEVZs6ycSxSuA2krVNWX8kazL1xgXiBFGP0iztBCFBj9k711QG4xpwyHNjmj9aNbmAEKzdk4/2h3WurPeEdpwKAQdpOSA/9SH/znSS/QsWHJZHtOMjLXPruR2iLIGjbzuYDSS6KuKgOD3klmvk/wuTHRwMcodbpuzfJwdX306O7l2bczoVTEG0xw+WdWkKeQgpnnMbi8uYBwVJbnSPCmVI8KRYi0O7DVloH3hHZ1GizyJl08B5gVdM6+ggL8GrIgrcDqYbB8+ssdCpEgigPuwG1SSTEfaz3M9zVzC8ABCVIX47J9cnk4DyLmx6ufKRmVK72/Pzk/xXL4oZ0Deyuv8R0TuD/7i7AA3JLJbVjoQKWT8gOYL4AuveiFTxkgtNb5rsgjM9ufj8zPQGoby97cXp+fTkX5f4BIV9ECyeTtiVErM2pqtZxgrbmyE/QS8kyHAxcm0DEwz77ZE2g0VriURbqRT/kCwO/w1mGTCStC+AdY2eblrC1YJbyA+9IqVHFsO9sfPFPTkMsxcdHdJzE5WLDFWkkjlIrppLe/ihPg/DcShoOBlwWB7yJyeUKwvaszrG3KcJCRjD5xRAXRnG7ClmtQbYDAdO8b7uBScAK/beEHxrsLSYVdN1gryAJeV13HjuAQ87edr7X4WVleFawEM4AwNqxgCBB3TBjS23arILB64dnKmQgAdWbPuJpz6otpFu70ErqdOU+7OMIC6bc9oZ6/nhJvxHHQctMVoPoNlK624IUCGI0jDR6ecD1x1LDgxKJEWKOjQ6DHy/jh61C4iYF13CFt4rfOACCHKUIBQCGt2BOFIAgN0pvFKu1gE4ggBovAF5YNJPLDJEH3WkxDsFCO+wgJXyC//iicKOUpeLjfq323MwdqlafMbQAj8gDnzqgFKoyecklcut4fmEuY0LyB4bvFgsdULFP4HgZaPCXjiAW1XeLhrYrDiXnywuPauGq1Wom0r0xuWgLK6GuZQgVoItSNFO0SLGEI06lk1qntsK80wMgebBIhmoFIidigOtswziN/LGk/l4mZ2fIL1zrwQ5NiXWaM1ESSpGB4DrgeQiP/AqOKp3sBG0vC3vAzSOMtL6kOyUDvVU6UMI4izVT73DxcjO5VRluZ/vvdtg0Q6zJmwMCdwsQiEu23VHn4q5bo028ZRHEIiCXlPCZZkTSSb7fJrr/rK2cmGpSEQEEuEJDPd0emRcN5rtpII2qE1p4EGpbZh0uprpZD98pW10DYAdTjk99gg5kINTDB83esP1sTNecD3ixpvTcSFGDI4OAIIX9zYxsj6eS0RVw1RrL0Zw1hccRATFqUySnJ8cUp+PpwSvJ6PwUd+Z+URARRhIsgJveOFaLF5qsBqxixtEu5HpMnkKw5qKx6oKYoSObxPL/Ka3eX5/f53AON1RS6mv4hqjeKzogyFNBiUAJMA/4sfYfmzj+/JWRqu2QU+GQE/qUNgJ9q1HQY/GHQXJ5h8TdobZOk2RrNjsqihYpmyK5EzQDIuIE+X0kGjmAoPmkyGKgxHeh1cldNF2hysiwZsJaB7mpS5f3DYgb+eeTxH2jShMtiPu6kHJXrpfRSXDlJYzLCxMoY1mhiD8r0olB2BScB5QxpuFVUcMynqn6MoY1jnNnD1AJj7/nxKv86SrckVy+nPOsyxlSLlQwQ52liZxpRFTuY1BfIRp4OLiY4UXZ6d1xOsAOnngdW4gRrURNOz6Sz4aCvP9ESG07sJ9+c53dGMoNnubWJsxxSPsYzDIyxEPuyYwDaR66hazQDB6ZVZ0VuxqGog5h/k4Afpw2RMvt8n3SrJH3JjBAJ09DZvahjNXzc9s62RO1SrVat4ty+My7d12FlXryGNqzSsIb5EQDBhMdhnaA5OMImWdhypWz4NV+TLFN7XFek4dXXdIhmUAW60s5LfFS74bqJuAvvm0XA1c1ZVmEYO2qKafv8orh/1vEdNJ7YqpiRpA7+BssrMg4TrEB2fIFBfzs4+nl0cn8vo0kYTVe7y9PcTLy/C+C892X7U1c7X1d78mKqUfoxJAlsl8XrDsw7xmjJov6napXaQGpoA7on8cELmfFGLvXkkHguiq8hXh9Kcxd29NOE9+XHAuPGYIvIXmf8sBt8gtjP71guuhD8uEifDBckBQ2vg1zU66TWfg6nRaiXmVH+iyfLSpPkMwHg6OfDgi/fe5VIvtvau+pqG9yxoJ8LO2jzt9uR+jcXLHYsN90TjrOpJHVYqjzqzswtoaHzzUi9vdJArBFkvSs041DqO8CMZHySID99STya7fK839aivmmVTIi9oadQLZ7Qvb/pTWY9HByO2jK4yVZQnIeK2/MuUMvD4sG+ZYDJr8P5KvERUyzHkwHFFeEL4/K5dDCpMqzTwosvgqCFCZfIpU8Obpo4TD0j801eK7a6S3m7JKsZbV3AYJDHItpqlY2TZUVPy99d/UOGP7QtPV7xOUG+FB8Oi9npaiZ1MUCpmGTyis9r5qm2PxIAuVGRYY8zt2rd2NbLr7mPH9Z4OFvIq2pdzfxmkh9YjRjG8XZY8+/bv0YsI6guGW24tdj7RNy4w3OZpSj+9k+XPR+znHuCIIkZYjq2Fo/97Ai2uWLPIrPYurzcyra0mh9/Kib5eXethhH7cZv6jXuu2KZAE+EpPhTgcJchxlTt9tqo4/4vptfe7SPuxEmPy/q88urQrBcp4ezduplmaGY8tpTWv58QMmydAQ1tykY+4X3TVL69lJvDxaIc5H7+/nPEnZWA7ccVzSEAwB0a4/2RPd3lYRmdY2i6boh5JnvLh+OwcT9+E1rLMS55g4eMB9I0luwuX95P2l4d3YGEdQLu6XVjhc/5EuyiyVonlDH6pNhF3aqC1gtsPWF8WVZOBy6i38qrquVTzBYHr6M8IVs9NmUZ05ImJ+L0L/Ya0BZhD/ZXMr9mvGd3jNzdRkxaVYxz/53vfuBRX4uU9OKr3nt+M8EJhx7Q5eRf6wZIgkAMwvMmh5tei/jpRwU3d3MltuVT6sLoS55sTfbzf44L20sJrbnl+e8xaMkh/zW4LvJaCViECP50ofiaTUjNe0l4chK2q+8i44nvGS0mfv2ZRLa+0BV3AsftEq3rNmozfOoFHnVzdetBEROmyrYCRB3C19lbX669B3oHLLb76L1cbgZI=')).decode()
assert hashlib.sha256(D200_SOURCE_CONTROLLER_TEXT.encode()).hexdigest()=='2ba096aefd5ac1dcd248120133e5b734f2260e79fcbecddf5a51f435283b853c'
D200_SOURCE_CONTROLLER_PATH=WORKSPACE/'d200_source_controller.py';D200_SOURCE_CONTROLLER_PATH.write_text(D200_SOURCE_CONTROLLER_TEXT)
shutil.copyfile(D200_SOURCE_CONTROLLER_PATH,SESSION/'d200_source_controller.py')
PRODUCTION_WORKER_TEXT=zlib.decompress(base64.b64decode('eNqlV99v2zYQfvdfQfiFEqbIadDuIYUf2sZbN6xN0QTFgMwQGOlss5FIjaTcpMP+992RlCM5TptifrBB8ng/v/uOnk6ni9u2lqV0zOKvm1lwTqq1ZaISrQOTs1d1zQyoCgzuZ6zd3FlZ4rmqGDRdLZzUil3rSgJuGmCdKjdCraHKp9PpRDatNo4J6zK2EXZTy+uMfbZaZcze2cnK6Ia1wtE+i7IfcDmZTCpYBSeKBpprMHYj28TqzpSQnk4Yfgy0tSihAeUsm7Mrv0mfZOqDKUjx3GjtZrzUaiXXduaMkAoDmVUvjguzvi5gi9cLv11sRZ2Tb3ya7XSxoa7EKzPwdwfWXfFwYoCSsAW+THMDVtdbSNK/FN0V1oKJqfUacml38oXTXt/9pTRj/IJkfUoY2FK0mNXW6M9QupxP02wQY1Reg0q8++l8/vzY14W2MBbceBY2sKpRBh2opP2spXJeZBTpAY27WEOGSt0px5d7VnZCuJSVR8RI8hvmRxFV0Oh7ZbTiy5fRKVoxqZjXQ3nySGCtrLVj0iIYrDOydFAxp5lWwPpS+6v5uKaHLIWSGV3D4EQ4gd4XtNtLRIdoixyKqeHZIPo90d73kAQmV8HKPN5kUFtglA6M64xkPWCOvIVG2ka4crNf/b5RhyAnb2eIlhLQbjXz0LZ8RuZnvBFKrjCoQwgf6doHeX/4BJwP9TwB6j3XPAXt9F1rURVYZhAN4kth9kqXNLrqarBZ6W6zL9rcgEGnRuH92FWSlso6UdcFRHKMLNHHl9CFePmKHz8X/bnXn7d3fJnF/GUUNjk0CobbEhTsHOGnHES58ThByyZwamcxGxKpTX9RCO9SmwrB/ee7P14yt0HEB+hH0KHoDvMBb1rVdzwbJWIM51P6zv6vK32KMMPlDW6S0pfs7OT4mI6EsnRfG3b24hhLC6tarjcO/UqX3rEVHum6ypiCL9QiQ04/3bne48v3fO6pJcFbKZvP2TNE0iccTiuJ1uPUYi0xDIuD6JTxn1B6py2owYkR9UWbSe9HGqeL64yKMnEePQUXLKIsXMxYDwTmkRBiwsn4WiIxug0wKxqiMaWg2jVQSMsKk4U1JQYtRWdFzT6dLNAH6SQW/asvTT7xCt/4SH3NvVLEh9NIh3hnp7PW+qZrc/YRWm0lHt9hgqC8sVmcvDZgJUx1dKD0COhKDwHyQnUNRKU+RtaA2+iKyNcR7tCyNH1W+0D9r+/wOfshapkMKv80QllQNe4DPkwrIcbbFvysmLOH3tiNOHnxM18OHYhvlzycJd4fTEFVXN9h76H5fAO3lVyjpsTDsrfw0K2ISiZWhNRonl0jHvA4+hd6DL0jss6JvOzApoNbtDJKULgQRlkhKxy880FoYcD5GvaC2xPMNp1sZQlBnJdtF8xjdak/8NWWt8JYSEaYDoaVrsA/u5RHqwrDGSDHt+AdzThpfbso7CyVkapf8GlIWDqDVRrwlCsCP5kGZTsDhfdqGBc9Mrylh70+agbMJDaoXyJY+xdoiGbX7+TEGukvbGDbrolnxsFl3trV8TJEiZyAF6eBqmp8BM8/fDz/ffHmsvh4fn756Mi9eLN4vyjOFu/Ow0Dph+98UM8kKNyfzOmoxqF9iByjF+NZvfi0eH9ZXCwuL397/+tF8eHV5dvRRF7JGgrCbjK6h2+2vZsXb18hrjG9r5FcK/YIZK8BSw2HaIg/Huf4VfAwuCeTO5ndY7O9SiP/5l1L4zA5kJk5OZCxg5HP+35NIz9AmZS6wRkLyYExQVMi4/0YOAr0fYQOHvUOcjqGkqf0ukjj/GiQJZPYlXEURGakPz3J/Ts6htwjNTxOeMTkDs/7Cf0GYcUYQJUI7wN09ajpERliKGj2H14U1LhFwXG0Vjjqe2FRNYiL8GzY0tJPY37/BuHfmaCo8DsS/z6sz2P/Ev0o9rRJwyFWg/lyDAY8rq84FYYvd6XBciGB9VF6gioKkvEh+yoOAO5rh/9mc2HW26tny3EHp5P/AHdKNiQ=')).decode()
assert hashlib.sha256(PRODUCTION_WORKER_TEXT.encode()).hexdigest()=='9a70d208fea6d08aedaeb8eca3c9a7a4ac0c43e5b48c4cdc9a43e470f487c3ae'
PRODUCTION_WORKER_PATH=WORKSPACE/'production_worker.py';PRODUCTION_WORKER_PATH.write_text(PRODUCTION_WORKER_TEXT)
shutil.copyfile(PRODUCTION_WORKER_PATH,SESSION/'production_worker.py')
VERIFIED_SOURCE_WORKER_TEXT=zlib.decompress(base64.b64decode('eNrNfWtz20ay6Hf/CiSpe0DEICXKcdZLBqlSZDnRPbbkI8nZTbFYKIgEJaxAgguAetil/377NYMZPEg52VN1nV2bAGZ6Znp6evo1Pd9+++3ZKu7P42XmFNkmn8XOXZwXSbZyZtFqnsyjMh47q8xZZ2kye3SW2TxOnSx3snWZLJPPce4khTPLVkWZb2ZlPB98++23L5LlOstL5yYqbtLkyudH+DXYlEnq/6vIVn5W+MVj4QOQ2C/zaBZfRbPbF4s8WzrrqMRqjkD5CI8K4mqzXD86UeGs1i9ezOOFM0+u46LsYRVv9MKBPzeBNDsobqKD1z/2PHp9n5Q30Ol4RWV9N79yPQS04Fr4ZwHDukqz2a2TrJykjPNeGi2v5tFoMcjjaN578/1w/+AH+svzr1zXG90MNmvEUI+qcUN5XG7ylXMzuIkfpHce9xUQmq2SWZT27qJ0E0t/VXm704ijwRxGW3Bhv4Dhh7fxYxFc5vgYr6M8KrO8CHquD/+NXM8bxKsZzE8PfjUav89hRCGCZQSYXSBExotok5a9O69CSLLAbkVlmffufLcECihKGLX0+G7AbwTBUiEpEqCFaDWLoc5HmhYpXyCYqmweJUXsXD6u4+M8z/JeCb/g+yAMV9EyDkMuiX0NEAxP8Rj/HsDQ41U5WN7Ok7zHD4KW+AE6FGa39MQQyuU6oFpIAWGxWSySBwI24N8v3QEUcXXhAWOqjB/K5iwkqzk0Fhz4UZpm99DTVfAuSovYF/QF8q/njbMCqGadAmX3AKpP3eepQJpPk1XcAzyVcSEIj4oiBgrndwNYessgOHBgDapXQBnreDKcBsErfp8CMQuIn4PhGxPKaj1IikWygpGoIgPoskwVfIU3ZbK6HnD5EL7N0qxQhScjf3/qQzGgsdV13DNa8vYO9gc+UF4aDOP+3/wcf+1bpF/VMyv2hx4snZdDf44zHUChRZpF5Y8/IEQAWdU68I1qrcUHjMgoz6PHUOjcJGj6jXWiAlhTmVxvsk1BpaWY2d0vLmHWHREx03fGtee71Lg7ItKlD/QCPvAydUe1ZcuFyuzqEfpeW4dP3Ol1HgPK7+IwXs3XWQKk24O/4hwYLK5nGQFw0f+O4zX0cZbl83juqNJCDQ620AeO1ccfY+fi/fH5RydbpY9URdhx4RDoZTxPoJIiJOTQatlXTce9yL8Cul7fRAYLEHra/ymgLz8FQ3Ox07sg2FdLPBrMsvWjzRCkzFCVuaqXkfcmEoyuvGgtEjMuAd20Q4XxHfKAHv/jA+FHaYjLzF5b/NlaW/KqWls/0Gtz/XARc/3AqJBC5UMDWUh3UBb+mSeLhRSjJQXrtI+rxqtXMcoMlsmqJwV/NPooH6OHnvdTUI3wJRZr6QHy4arpV1N/0h/6w6k5DPzz8BjoQsPRq+lYIPR6D48/w7r+L/j3pzc/eFJPIYdXXvzvDWxmD4/+wyOsNGLg2PIKV6nwOuDOSfkY5tl9L48XMbDqWezPNjnybH8GjD6P/FvgqsRZH+zp0hV4eoJA6vFjY57kqzlCkFXKSFeruqhYSb9qovnR5hHcV3ckfXax0+6Iuu5S390RDLzHw/BdmKcwuircEUFDsDhziLmrokfd8jzPl2mA0rFZMo5WjaJuvjShFf/Oq7JU6PvvD6AcMJnz49O3x+fh+5N3x0d/HL0/Do/OTi/PD48ugy/UnisCHogMtKDSOLqL54AK6Hrej5cbXF/9u6HL3XOR/+RQOF4Vmzx2fj84FsmsGFeCn8PVQRzEabmP8qWzWcN3EJyA8ayxxaKEScCffcahw02BCDOG2gX8iPeAoUHd+R5Dc+JoduMscig+ltKxc5WBIMcQgAvGUCN2VrBZcznd6cUCt1joNsxgBPLcLFuu0xjq59E9UOXsNsb+X22SdA4sKgX5cDXP7h3oI0hahRMtADNOeRNXFUFK/Vc8g24+qkZmEewqqW5jb51vVjhYYDCr63oj3ABALnlYCu4ckVjmWapZvAIPexduPrjX5FAMgOOCd9JkEc8eZ2nsO0ht2BwhNs9mMSycgmaAx1HND8w3ifEzaBomO1lG17AfbK5ACig3+BVl+tkN7r9z5zqP1jfJDITjNLou3IqmLi4Pfzl5f3L5xxaaApEKGihv8hjUCkB20Qdwj/34IZqV/fz6qn+VbaDH8/7dgRooFQuBMsLsCvfGqCRQrxQe4jSe8Ss3Su+jx0ItS6z4auxEm3lS0sPB3ivs/WMBUnbsUJvpI8wANumsAO5d7Jz/+gvsnrBXAiqB96Uo7cOkLLOidMp7QFGWAi5oJy+QXIXY6KfZP+kdDCmU1R5uYPbeuKOhr98KSkMCGjJQGqniJwcChrhwmKzWmxKGKV2d3R0MZnflEfUI+n3w6/nhH76zWQnYMQ/RGHI0m8VrEr9Jnfnl7PI3tVg0WaEUEOabFGkLZrgEWolFyiTa0TIHLEHoxRzQN3/cu46zZVzmj3sMbi9Nrm9KFsEKZ7kB5F3FVc+gI3lWFLS0iBZoenQfFjHgRLqAE2HgdY8w0VfMwcGihHtRT3ENQF9BpAZ6Lhl4fcXED2vQVoEmTo//YaIE1kiZwVSMHCFCgyZacIl7LyoUuJqPsVdqWYHMCyjAbcP5LUHOhVod1+8jeS2iJAVeiXS6ROajngfOaebw2gNusipgipY0aL/a7PrVktaEj4szmkdr6ivQfp4ALFcESqKZcJHALgBbbW9WPviNrfZrNtYg6L35wYf/vfLaK7FErivJI2xGRP5bN14oMPyxfdvlT9xiAIOYuED97lTTvyk/VF+Pzt6fnYdqaXD1q9bqWujYXllGGylcXNlIYQkoUoO+ah09LPxkuVkGKBC07/xjWSaqyAzIsQxX2epznGeqlC2AtPAZaYe5TRunAXGF32pJwywJG6awVXmzZxBG8jm2KqH2nKxCWje6ZVAHCB9S/6fgQNfRy6je6SZSrpoEEjVemeJSBVsNZQ2bTlpo4DY6o2+Cq9batGDd0VWWkbRuCrWggHhKYytAPgqZUYTrKMlB24DFhhYOXGuLLAU+6afRVZw2F57PzFTWH5dlewb/9sYxWj+KYDJlM1VlpuECez2C/NId4EvXUwD/A8sMERH+xbVGMP7agqMdJAACV22GQG/uqOqCK5D4vQJrEifqWVIXNgYQkJIYiBQJIU3W9tr7fnjwxt/3D16/9gysEG1WJFJ1BcfnjhhTVUeMt1dPL5TtEO1WPg2HRAoa1wCmc1n0DCURttDqoTJWUHFQwuC/UX84Jc0ZX4m2+soB+o75lVUbyuE2aCA5WRIR9dBoUaOh0H2JnQRiWq+uXVgTYjRhGhxEa9x5elik0hDjB9w+nWP6B/ciELvhXUudl+7IcV9is/Bd2Nduev7y/fdM0qBH4c4IqxKVi5Dhu9LOk+iTrBiELMMUq2hd3GRlL17daWtqgWa4L5Vgw2XdkWUsggrA4paD67jk7z1vIDV6qGyJ5OOOvjzZ84szO3GhhyDKuA+wecM/KBmF8op/83sQlNRr+slvSW5S7+VhnuTu1CYRHshE92Q6weangTUMsnCxXQtGQoZaNbJ5VEY+zaTXmMtDKJhcbUq2vHY29cWN7kBwia5QTCMz55O5JXE1JYMAFtM0rKbnKtHSiHC6ThaobCpQFiek5zbAhAIfOmebp4nsm8UZsDv9JuBf9oLbUUnqjDuKkbKBfA705rxhxmIERfemJsPcEUnNVB+mY4KkGD9tNCXi2ixlUDZJNqTuBt2LYMwlyDIUsLFgK6lP9qfVCEhCh/6M2U4YamEe38n+V6BxcV4E+wPLbRLiumDr7SvPxjYAy0HLDbBLA5DjFyHtzjHgbsysr4YuHojd3su26n0BbTX3nfNRDKysZouATwzeQTeOqTSjbghKMezveXSNUr5S/FBxFpV2YIEnHCmm9+V2ZKzAO08sm4SRW/8OccIGYdkEnrw6Ziw0K7Dd0wsCJ81vBQflES08qNkA4YX4VI96C3Ps84+DqefLu6HxbmpPGO2TAPNqMpu2SGZtb6nVGbZJpH50+OH4/PDCnU4twNRXNcgeCoAtAjKwK4KG8yRd8fxis+y1S8otpQ1+B/remqW8oE3IU6iYuFgwTzK3worxzgCotKzgC3D0UYvmpUHCdwMYPgmWomUTT08VF8nuge+m2TVqlCHSqztCWzPxCvQ7JMtQa8AhbI/GgvfFbBCyqq3sJyLTMDlpYgtxdVBJEYDRalqnyEpqZq0d5q8MyywEVSCfh9rGSRMLZIaUVi/UrpboGkCHVidpuXc2MeQm6qW2tTFstJFG2L/7LJQJVmpARSwWIhULAnnSYkntMBvKBaL1btJQOqZEDMQiFP0MuOO9GtI16KYWJrAbH7YCr2iNaEoznezes5gIEKpvg2iIsqzr5rewn+/YU/d6boFeXSZCJhFDaHdfQmtaJKzLtzJEW4+csv0fRF/uwYAsNY3uEZ/dqr9JdXYjq6cChrptLfuNRhxU0TNYTPCXb3oLcEUwAt3RHXxBYy9RRqdN9cnyn+EQW9cmblb4ERqcbKGSqSpX0beNIkNE3zWLbvMTW7Z45vwvzxmeX+GphkVtv0sztKexJZntktwwbtHYIlo+Yd9mmxwZCGGMbRY0tJU9osyZZsARa/svRyAcksEHNnoOQ3AvYOqk0Wj2701SJKT13CXkdyzYfIyY9cVkCk0LJ1EakkLCwKDl75xDw7S4TArQBkA8SQpehjCE8ibPNtc35HRYbNJU2ygrx4PvgNTunB7/fnwuZk2QGowmzlZogl1mc9KAHeKOYm+Ehhb4KkMHztghB6J45wBLCB8dB49OsYFeqe4N6k5atQheiByt4l9qwm3QlGT9LUI9x250iNoB/d1YcF1UGiilojLdtMNtqiZAkUWwRQlhyd+fiMhMi4UXz6y4c6c9U6dpLpUBFvKxDc/WVJ+1ZJREYGCUpIOCxQMCq80XwuuQHTAj0hLB983CBIVYRljbuisG7Y5QAssnXRLA9Od92jNyUgO6wG9WQhaKWWmwXdv+8+AapkahWwZMzHHSvTdPt8PONmUBnJs4qmy4bVA7dt/tXeZ9kDm5BbZbSnsOWCXRtADukHo6oSpWjA4cIT5SnzCkBBlL4dxnuFWiE3KWbsjjiW7fzXpvffNYoPeQio8x7isGoRfZtHaIZOQnzStnjvtUNxdtX00hDA326kfZeHgxeV3mCZwj2alajbfi1CfnfYsNN35Yxxjb6IMO5otnmadQB9D9STus9gOQZnJ9fwPl2FD5TRVUtSDK/Wt2WuhbN5TnWmpnJWhNwVaDwjglERlfbzFTyMDQ6hc0QyvYkW+FU1gqD73koLEqgGKHj6GKo2hXFpAvKsFKQ32WdwEbWD1yA34EQmhw0O6r0LpMs3dqdlpJp/3LX/CHqNa+CRR0AxZz+FcHgp9qDwBoky/uI8N8BERKxMuD4HQVp/ziFl7o8At8oYl08ug/+LdTrCCRE/hZqE99fCKORA+kKPOIJqNXB9OpwfMoXkSMu1sdOUS0egl7XheMrRE7DKVfQanAKH5gUrs7sriEGm+tTLVUjD4ZS0ZGZYEKgpZalYcbGPa7w5P3x29H4s1GafUapVQjZsAIkEAUoQQYpciRgamTtlGBY678nXN5nzkLQNiN7AHZgsRU1OkcmT+WhkfOPImuV1kRs02YuTVFX8N+6nBELO4VZDuhKMZiIOxgHUcl2R9tl1jNvniwzZcSXcPu0mpvDbM8uUahFwQ1ZEET5ja2hUr6oO17mq/cFSIcmYRbIxFsurmAuzYFz1bzm3/Mtq3l9Kxmu7ec3Q2r2FbL1UDteBP1cfr0tR4i95yQK5pc01PE28HEtBPwdMiqAWkwkPnp8CyRlUACDm291PSZfue8z4oijYvCQT8TCqSsImE0GSyDaxZwgLiXKPsocQE0r9U8jQcd3r5Jj1qv0I4OtkqU8Hz+zuTDH1VoRuds9Dp9mwzgq/2b3nTL0slug+1ORHIm+TU/pdfmjMxu63N/BPsDfCCYDjsh/1MeRkU3wnfKZKZch9OAf/xpcnkBivbpZXh58mFbEKWWTfoUC9K/idJFH0+aGDGUstNUYowElru02bzSkXCkQYBYq2KjZhHGVWHEo4bKZgHEEMc4pY+o1BPBUiOVTO1QIypGEuMNYbBFFblWbPJFNNOhYPN4BrRMJ3A0b4cFh96UFS+TzRpldmp/7JA1pKxaK3Df2HCQJpnjJY5WtYbBkcD4VWsSlmlU/znoahd+3yXZpuhLkGaa3at+1CO/YENblcln6AHGqVbQx0SYxeaqLxjHb7gPYadB+2ZHkI7r6l9FBYZcxlFRJNerJfREhVtxoB5Pggo4J0cUaAfzPxtvLq95wpCdYPA4LV5NIZ5feweo9aZbDn3kpdeMZYcuqjMlefkTPBKANPNvksDsxIAaYih+8wPCMZtOs59uEt9x/8ETBLJjCuI+hgGWTbKE6Zwl5FpzTYWNOmHimGcbBLPitgvRRsc7p8RswwzG/zlIM++/zDcwBs/2ZzNk7HqozDSoN1aHNE7j++qgHMfoYZjoijyLUDt3yBr3yFa9PPsMC/jGCBlE/QPDHPWhDCWmhBxqyDyZIohVZ+i9Ox3rkqxgc0kyiV7QC7dyLHOdLiR+FSINZNa6qioQKNE6U6AP50idezM616v13ugKnVbjYOleEaeLWk/Ns1LwdUCGDVIZaZ6UEHuKAi3F6l0xmHrBvnKPgzT/0zDuv7Fa6HTVNd0LPOyA2pDYb/IRNP0QVEJ4GRUJbMxB5f5guM//7r/22iEofakLwP5re7asyoQEUgelB60aYuf6Mxvwmk3A1IetRLRzZKbbnsJo1U4o6KUJwTVEof7VZsh9HHTg2pgNdcylxgDq8j+fVDQnyjeRXrd7NxemzIhv0HrQtgCE0QANzmUJVBwGWAZskgXaY3y2sgn9gyIGKDD4B5cE9a3IAG/xbIMeCjqj219hbHGafOagag6RvyYPBvDkMltz8PtAw8VjvJqRrchYlwDeb+IUN33oK+4gAKoQ7kaP2ADOGDA99CeAhgkysimbDOoYIxQcS3B3gKvUB4mTf7C49fb495OjY3ojMWSZOPhofKHRFYWpibv/QyTTwMgcrB/dKSJQoot28PPKoG+cMVOhY9Og5cxeZ+Gv6fOVdIjmxegykcYVkHYohz56OajqsH3aDgueG46lkgITVx3ECbkL0CF1pks0Oa7lBUFXlVDpeNRIsY5ngX2Ce4DvQpQUQStGe34248gdlIpJQQk1SG6tcH1p1sBOHSojhuFiCz38yxvj3wNcJ4LPkMv1ZGVwJ+lsQA0TvN2G/A3wQKe4+WCvJbzUznBy+faz1AbSFHQTVwkGKNJ39iFxAD+gA8DnwMzc5mFBgazBNqpMzUbrX3XzIHud/n7y9uTQkS/6MA+fZFrn2VUs8pYQYCA0WsBaLQWphYxeLwhViPgULhnEnuJPYj23hjwHrgk6RTIHgZViBkX+yrKSJZSP52f/9/joMjw/O7sEKSaP7gP8uOdixOIePO6BjluG65sQfg9u5ovX7osWfEX3gBcWef75EeAdvw3PD/8RXvx2WE3HMi4lvPn49Pfww/HloWsJzPh94sarOzpyDus8cN9D2y7JDdVHperh9+Hg9WDfbQdyex/l16B2woLKQGdDBTSA8QKwSMIV2srK6S8g+/jfWONgfxf0FDVZcbIAYGDchGieMoYGakywpbEUDyBlq0WCb83mqu8TlzRUHPMvhxcnR4yUAqbYLIMng1CHxsHCfHxxcwxoFbNhlINQ2lF4IiWt1qGC0ezZxVH48ezimFumbwpXZOrgoaPruAGDA7PQus2+BAQHhJzODVjAfOI56npI+lyEAg7cDmigLWKh/tAAwR+iB/xgvL/OE9SUcZR6NL+en3x0d52+p9rZptRg/QlKW/r/8j/ZY3jhUoYEXkEypXuwryWoa+/NX++Tf1K2PXyNHg82c7SsqgqixXTorcFp3qLkkK2XZPXFj4rZCIOhdwElT0C+URhwTR7sjalHAX0FVFH3MBMJ0AeGhqoPcsobjbjy1VI941WPqkKXf9jXiRGgErwY7muy5TKgCs+T4l+4h1MRUZGWWTVcfHL1QWjKjIKGCqwOY7/gU2jrJAUlIyno5CrIYiXHVqDSobBPVQUjlH2EONHNa9zmB+9g30Qm5rt5Iw/JdZ5t1sFiwhzRfYlwVFBuYIZ8YzlxGRcYuljPlDDm8z1tleRLo9arg0o2VypaQHUGGF4OUgtJl7TvGzpooH4M5jFvm7VcIOq7TxZ/j08zqJf1PREnx2pztQGJAK2fMfMZIz8EP8hw7CNc/E6f3zKTSvytaRpRIIzj42SyQJ2lCOo5O3atZPiU4tCuByiHV2k1hq9Gw79N2U849PxhlUrjRyuVhknc2Avvp4pA0TFAfK2g1ajz99A752ozh42YjKrxXAKlShAjWURzCpg4DHx2ojWsLr1kiTYvjo5Pj0H+/nAGLAtBcgz82fnJryenh+/Df354Dx/UrI2NOvyFXuQxxdg9LNHTosoaAjbu0u+xATU/mO9g3xvrj7+dmB+jh96+V+/i5fGHj+8PL62+iiJzwomCTH1FmzTF/iIBCajGDEihAjGGlaY92JySBaqfyLdwDYuoI7YcEecHYnt6nurGLDEukU4Kk13T8lbHtud87rVw93AwezDJq2SBUlXFqzUM8ZybDNYEb/FYK7eMDYBZHYpqsEERBQmvtAvdHcS4k7FgR3vZbL1xgfAoQck6WeHp4aOPn9Di6sNOggtiVjLWf4XXHH+mA/gG9mCC9vbauo20VWTpHTnvl0s8hSEiILQcHp19+HByKYJWVUWdIw6LOKaB/nDQDnudFSEeJiaxrXq9iq+r1wOrLkVlqoK38JmDlW75jFCRXC8jqYonx8pssQhvPuOhnzi6BeEWhDh6xKiIcJUlRWy8BNkF6SkDER8kiSQDSlOqJUh01kpbRreomN4lecaG6mmvvjJ9l/YrleTIdP1QUVLbHmtA6BhG+9GVKoiD9ivYIeT0Ue0Ukq+j6ER1QE9mMcvjeOX6pOD77nUaQiPyaAa+XqcNW94MUI4eAd3OdYqpNmD8Yp8RMN7oGtRJ471xTkclNxAvsTyF6OUrMUh32lJWGS9uonwOsnQcAqLmSRQqdVdpSfq7+tBTEDxfYYBsogFZ0a/TKt2V7QVUhZmdmKVFSQ59Qez1ehNqnSsw8O+7bSUUgkxE642ljIpbWuCiXIS40FstrxoxpPMB45pnIMdAzQ9xEbGQ7abp3XKdgPiL6wF9dnYlZSEAYsfUZeTTMVCPAsJIWnsGdrX6q+aRRpXkuAEib1oA5WCGOSQpStamdz5iRrN4FduLiqQcFbRUp3jY3MJKoz0GrjNAdoeS4Oq6sfrYAFMWwZcnK8J+BW0gbhS0AeWcq5m6kToCLMlYI7mr6XSFQiNuZIKYJjsIOVkHwjJhJ4BtUk4NWt5X2IBg/lBIA/VNbQgj/AEUhMcfJBrAULx9WyUYNbUH6wQDQUEMo1AQdmTO0gJkq93FhhejM7dE48suqPWpeAZ0QiPJt4UrOMWofSDlFI182KI7on9812CYIarbuKO7o5rFwe472/Ok0PmnU3QuI4NWpEzHiumXL8GgJLfXojGUHCtyfGsZJdDaB2xIfcE9ZnSwv+9rjZpfqIMLxJMK6aR2fagDupIubSJZImoDBHUT5hJdt5KypczWfZ5d37nPI/IhS36fO4y8R6b+6CxS1JmBPxlGVa6PYneUiwq1uYqdf28itDejNMtpz6LSznNGErPK59HIoFb4EuVE8jBFJKmMPjr2Fcmqist26SQhLgnMpsNy9AYz7aCBPLtfVW0AkY05OpYVRJWgrKg0Q9YoUfi0GxXrtZKGlPzXWGSmpAcEANKSrgR0Kb9aIFtGb3fUEtpQW2S0eehkQ2EjTL2ZYKoNQBWx+5w4dxOA5cDAnDicsSfKVxxfQGjOlPJD+B47i02JSao4TSmuEhRHZ5KgBubM1s7Zl+FWbLklSqTijxIdUr2omGgVUyKUwIxigFqQZ2aU1ArRuKrCLKGrRmM34QgkBJI+jlDQYo3TdnRsO4htOsTxNTMTX/iFX2m9xlDZJUEGMmV6bB6gQMkUxYumlKllViWswo46/PvfX4uA2Sl/vrCSJBTBZGvwsZy/rR2G5XWOMvlkMPT39X/9wcHUn+z7xkt5Y76b1jy+cgTbPluNfH9A+QbH9KqM1+bZfO5C09Ji7+NbrAldrfW5N2QcbE/JaeNPxVI9B4ueESoWzWLThsTAtF/JtKJQ2XqSQcxXF4EMoijeIqOQqgxWa+CHXLseZ/CFNsINsDf34+HFBapMgGA64OqSVjdCUvJ1DBCooKDMUeQQtaQsQWQKUQxE8hkDgaDJHyRAwKMwArW2TBkZMGZqI9a6o6Who3Rpg+1tX1fiODE4yejrVq9aaWPJ7RXYO/VYAFCTGFErzyrjwFin3lMyKcnE8DQbTabGaW6G/iTlUdJQCQCg5P6gWZJ5y/UVfodpj5dr6H2vMmP5KrVUtRo4NrEDFJ5WCWxASvYZvvE6jJffOR/ocB1HW1aeat65OcUZ4mNPD8uhJId0hkVHKm8KfBYYkhdQQpWZYzPSOS1iSFaLnjdu2UFwi5YqKsCQn7wtjPJPKPfP5aVbdHrE3dtq+fCBHkdOQimpjSPnisGLtmAZnryRJjAOvqAmV/F9qN5bajbWJ+OlzwFiACdebRAQchSim+fnlOAweROHIkaXyhkkFORzk7wsQboo46ssu90/CDmdp80/FccsFMcU97fKMsMwhf1DOUlk2Xs2i9/KkpmVaBb/91YWD7MZbJlaiyeojWBLVo3aWVmDgezOU+F1xlIJeQCHIMKY0BxMA+huM2YK2C59/j/DfcwxjKyAy+/tDfe5TpWkYVeLzQihbhrqCvGyiRqmakUkqrIPtx2tUgPbcrbK592dSL8lpipZqBCoujXMzEHM/Jv7pQKmqw1borAa3LKx1esB1/k9QX5+FhZj6mTu0KfjnvEm6YitRjI6UsmRK+vR3XN9Y79YpJvixkjn3ticca51skjkHGqSdMqDWvYfkABWm3VlKpPyoPvyl7opUqUaUrZHKeaN5Ied7nqboLBFy7DWEq8yFYteLTztdbKkKVHpq1hnkaUIf2aNz2g2x1CmAviVqq0MANIyySoA4LMr4ktgSzHKMeNTiIM6TBrgv74CVfUkMNLJdxyariFAx2fSDkNHpidf9IlAbdNkvq4Sc/ACupOjYwmnrah2EJPxedKPMgNBC6WfMQdDmoeNujjHFjT2KH8F+toJdx7TchEoPlfN/XPYDsbEUEwmOu1AQFQXTBSSRwbIzHhmX2olI+37P3hbvLr1nOnohGYEEoImeKjDtA5yFCSCgq7PbnuTzohkAlWP9OT1uZrzlMwLAxVim/860BJ8ugXs7mlScfk0U8LbsQE9WB37qbqozupUOhHRDws3X8jCKe47Po5PRlI2zQiNtnNUy+DBgw0xb3RVbwvH5RZ4rYQ6WfRmiafzMM02+4Or035VTiu/4pjI1lgk1nmxHdwV0HyG57rjQiW9xY90YoKwborUrm3X5mPUenLUMsRpqvNiWnvVcTsrbQqH9spx39oR4Cp2u5FDROVoVPPxzADnOoE1T8mhhzChxArIAzGXzI5DthSlzb0H9teEt4qvoz8JD9foU/t2IIOqbweE/XEn+2WEUEd0rgqeHaVSHWXLJShKojdLgP5IM4g9dQzUkDw51Lg/9EvfAZK9RvdEOSpfDn+cDkTfnSczCWppufujP3xdMTK+VqAeHCEQvJ85ak5aMViK7FeqXY4jpwt+uGYVHit1rXgRBd4f/uj/Td1Ig84jlbeDdk4yRkjRyqRezYD+SHenKJu9tAdLnH9o+lBCbahwy7mpJgIEGLSviyi/p1Vi+rRzL+NRTPS2VbU11ZvWpKdbxKtbpuNGJdW6UUeqYPlK3kHTPUbEZNFWVsyX6qyT2W0a8706FBr1eVRLI2f0guubvf+MSoF60jg7qPVe6lUDgGr6wai1U35CSQSNWbQsVJASC1Dff89NSgTI81LRcFZd0SyAzNYDcu6pe6544yMVmSsMrtPsqud+75JYgJY48l3JVR1rueSIQxKYE7jcO5dMzUIqusFJpySq3Ye7RFavJWFqezKcHVk9pqOqW5yR1O5B5cdUzpLAMBUenX34+P4Y44KPDk/fnrw9vDx2/crBWdnejDAYXwnTKlG89r6Ij1yHCdCpAX0AvwJmApZMFna+f0O1sWVvER20Z8/couQb8fCq0TbK44KabXEkssxR44KF4ZuKkdxkefIZuRIKNu7o1Y9GdgF2MabqIOTIrTHzsQqMUkzV2R+9AWTjMZ0Qj+lUw6BMTy8MxzAPk7caYIUkVlWEoaQgyhNkivG+S86+VbhFIaDrUbCQXdWrZ1uoSVC1u0icNWigcoeBJBMt7zN1hwmnQ5ATlUzEe5wWbHYTg4RuiFGVsRGLZxsMy6dl08cFy0ohlFfnvckmvY45jgpNkSnfp2Ee/xE9yTWydxAN6HVD8zvfyAzOM0wYJrl8fVe5wpAw8CYC/enJWlTAGNW5IGAtsEnnpM1jvJ++0O5L58KpoHSUmO5eRbVBPXWllq4Fzfnq0fJ4qZcqn3TLuh5VzrGJe3H26fzoODw//p9PxxeX4cfzs8uzo7P34W+HF7+5U7mHim2u4gPoVeZ9HikGJYjkqzmVEbzX0X2vM47PwClzO4zZaeN3fOhg5wwEgdXV9qa6iOB/kwqqasbbqaaI6jO/mD5hINCRPsWmvjtGj2uh6jqTA/QDt6oGyGbWy3Vg7D9KeFxXgTa4BedxKjnYMpldIxCHZkVv6EEATVvdRv8eHUqq9bWVgLPVV7udhEz/gvtJU3GN8BXk1lVVyQW3MefM0BuvFWlUiy7y2+OH/G1hQM2UGlZYTyOepyOSx47AqQXb1EIv/F0BHH5rHIa/JdDC3xJDMW0cJGtfKxPANa9vJe3gC4wa56naY/pRplgdPce7UIHpNqBCF79terZaXGJe5RN8GjMO8nihPY/POGlt6Q9CY02TW6f2oHQTlPFV3Hz9EsJtTTxPR6lGxo31LEVE2vNNNcOyvtfSSLGn1twkVKe65e7O/QJFwjb4cnjCsBRTCkyVAT5UT9rbLJkG6DcTDvp5e5MpJsXsSgL+NCakBxPQIBlJKhUfqkTzYEJYnFrZD7pS5ku6FuUIe60zHHB2kzjkTD5K4LOz56ODTDVglhiTTSWgXk4MtVVl+3qWM5HtMrudiVtcgFWXtngCdS7CbmegsoAxDbV76Yzhv7Dy0dOkVxY5nYuPbHBt6cns1oI2OmOjlW3srN3P0Qpt0rjFY7q1gZbyVptywcXWFvUlGDta0uWevBdfmTQCc7yb92IaphPEUuUyjPhqJ1cuxxybU2Pl67ZS07fdB9bRBIFUwe2qRsPzRQmmdUrWb4L9gc69ra05Ll33Fsppll0Zvr0/na1bRJhnNdc57F3Zu22i78o9uSPz344FYBis23IDWlT5tVnDDdavJrUtSzxIASfGBZJqg5GIF8wcyIfICp2J2kyIXSW5ruhfOOhL45ZglSvmqvj/PdhFzlxI0AsHCctRMVpSgxdfFwyC9xtsCwhp71lg72TjzuQ+PZNxU8oLzLDbq22EOE5/f2DzKLRYVwkVh6bJvbYgJTEIBTvT5dLmjahzutEDvU31G0wLlfwoUgG5BH7QvIYnmt/RGWwzdwdPj88Pos5MsM9o+TYe8SFeN9gVvvw5OGhyFt7mCdBB4+NXJPzRagxe/kvKgM8GMUkEqKU+X+UiMXP++DVBEanE72mpUNUxs/w0qgwbl7Zs2V+4b4oFcX+ROxrXLneC4pTyeIusXN97dPG7U8xu4mVkJn6mDHXIMHBf4Kz8tFdt66Ladr7cjiiixTg617ztYcsVh377nYbbclw2eaHfeV+h351d12/e6vG0Y5PeurP+bEi3eodtkZiBYkZbM3h+51xg7krf2axReSDupvNY4sVyAPgGU5OpWwg4KzcHyN7HwIUxQva6sWIbUUKruQ6EnhdCYd6Yc4n1GpEA0DLSHJ4vUyuEYxd6W/NWWTyjJV22pL+0yLztrm+ifM9E8va2eJbxKGivJzmPOd8xazqh/+g/1DMUb4f415VMjAwBFVJ82w0ttu0P5dODfTxfl0XzyvquP3Qcz6eEj60JvYwkb9v+4MmdwE66B+PHHF6UlqbHzfTJvbsQX6+ZbdCrJeiza1P3/C2Vd/YP9Dia5kBlje5N6nMNM8yUILP+APNO6Yi5KZVAAXNP725P5e9R2cgQP/+lOjHdWV0HxNOypWya2jmkQHvqCqYf3uB5IH45Gf3wZjoAkk/q5906eaTEEZBn5tlRFBZ10EWKnOeUUraaU/O0faE0TU3cVmu6VlN0l85QhAzgJdNvYry/4SbZNXTXyjwqRO+OJirErLpZyVoVGD3Cg56awvqOQ1atHVCbjNwVLvi38tMbSdqtwtDPnTP7pZmsvT2nu845LcFhBnv1b5H3mXncuQznBpaPTzu7wtfktXDRaaVbqSgezcE4tEp0psZXJXAZif/qt+61xvY2NthA5xVq32CPMKfRrCR5+P4mgw1W6UhiSZWsu/N4lkb63iJoQUvGlGQdBOTmXTtWUDdZ2ZSgqOSKNrlSWRPHZpVhvYopVrZG/4oArWOuKW73iFVCO2xXCf1YfOSSLK6Dd3UATyN+19RNRV00rZDiSmV9hmJBaQE0DJXmLWW9tT8jPs0jb7+nbHdUoHhSrI24Gf649vxG3OLMM+95OkLHL5JV/iixKhIt5RZyEmlPnO6O9qnbl1JyGvuipJg8rVMNthurt0WiPC+whoNnSztq7/OkFug0bZEeSkojWnrjFuumcRfD50k99mk6SaZ+LUTLa9XN7m+S2Q1fCWGpWGV/aKlPZZdmpK250Mc8nIgeeesPR8N9rVbC499GBz9OGwKkGUH7zBMdn/XphNB9Sd2n0TYMuJL4B395f1Ut/c55hzeHScRmlfIbKO8qu8M1i7wHmFYhibZhqayzHO/26dTXZLV+7oq4gkGpmDmbW03K/oF8+grARhRWK9xhK1xKvk/uity+0YjUrImLige5ydGeqvwQuyt8IxXMjCBa/ADdoLpfadvlUtI3z8i2jHd5dcJ6zo1V228nNpJ4yUZqN7bDsdQWjWW7lyY7LuAyvCjm/srDzLuUXj26an5s9wRwYj4DSglirCEFbUgN2ocf1DoV7LMiKpenuIb1ErNJGWFl/O/OyC6Jd2p8tyK4Om58Ubm91OVoLU4Xgw4qp80WuBSLxNaRpRVoJhXt0tU5WnWjAkUjYHDSEpNZYlYiR92eI1cl6sz5ypaAnim3kRlCX7KDeb1MMti9aL2WnAbPAGdQEluKGMcqO4wOhpIAaBgH3hyrSbIxAsxCwBBI82qQ837LulcmKp6ESmWrCrThqf2iKhlppzmsje80wJOsWjN0jYZ+7VsbfIuiDyzAtkum4gl8YacM21qHvlutWbus3GzYXNFWi2pxb4fSygJaiMnqp80gFHk0p9AkLyuLjL5D6avzXtBldfrKWPeXPy6Pw+N/Qilieniy3iQ4lXXMnnFhZ7+cfYIW34anZ5ehAadl8LbTsNnsX+C1Nd5ay7TCrMYQfx2VEa/w5S4QDjDZm6PDNfcpk4pOuKLTeVSXpHYc88cstNmm7OPCB1omWyjoZsnS7pFI0WI1EHFVyXFmDCxtQ1szh0iom0Jq48pC3FoayQ74dftRxw6PkZ2Qse10h5oZUXGYv9UvJFWJ2tfZutd9e1btcOSu83yy/2hBlBs19p4XluLJ3jbj+KRSW2snKJ+TkQG3IZVXW3QWuSLWThPO+XYxTZdE6i1v50ne4wuhC76bltzVYXbLiux41yE+fbcrRprrhjjCnJKsjWo51/wqbv380+npyemvru+KJkepz58RtUsJMJB24zVgeo6bxf6TzuNNMTtr2MfjQKWdCn85OX2LTbVmBXxWWphgW5J6m0i2x9Ea2YZrAaIoehjiJKO2StxH2fmrUP7q9VhsF4d0dZ8yWqiscyN1Z6hZpdVY0ZogTxvH7JMpjN6zd+/en5wehzxid6wakpGpQD1cuX8taYiRLK+9l6vuTh4dfro4fC99DD8ensN+5NaO/HO3mVtgb78+1LS20mS4sLyYVcOiax2PoR3We2HFPHM6GNhaUOlVp8mF3cpF4SPjACTsAuycVGrK2Clivmy7X+0f2lU2cNt84SQDIOdJrNgJx9xXMBN0SklR+0Jwyi1KuXva5kNkg+MPn94fXp6cncKeDWzg8vj83TF8AjJqnxvJoYMR2Mh/1H3NknFDn/VV5RQWaz4j6cOvHz+F707+CQIDbPvYh4+//XFxcnSxtW1oqzMtVNuEigDzH9BxYeI/ylXDckuJGZ8CQ3KilivXjVntWDbd2S/V2unC0/jP4IYyXfJd9L+jjC330H9aFZs1ygLaptznnHLYT/Nmec1e1LrgZfHCuFrvF+h243q9yqaBCVO+mBffoytfJ+x15slcgmFoBj7HOkmOOumCekIxgM3r/dkRMJW6nNtwxpttmXN2l9BdKgV1gBQRX9rI6AoLFMLUKKr4I9Xs4dH/fDq5OKEZIck3PKNO4MGvegf+I4FPI/fo/OziAveyo2P4F2pyw832/rc41Mh99+n9+xC0CLxU4+z8jxBziuDA2ff07uTtcfsU7MAAmwokDrx9qE1lggCf2cB4PJI/qROeeMqOTy/Ozt0nynWqrnn0adXV6V2Fjyir0OEvZ+eXoF7U8s4DAP+/48erLMrnJ8gh88269GoaiWh+tA6ujYt1GyZkXJmB7hdlK8OksoH+hWFeMG/AymbKsdKWTQzYQDO3MYjelOK1yfDkXAZIejXbu1r69RrABL7muKkJKuaL+/RZuum2nC9t2a7kDiGZpeb17JxVgkQzuUuBa6CjAxZDJZeRuFxnbg0RrXbbOyBXZVQGTh6GqASEIZpNQBkwrJ4kmBSPBYaR3GE8lWXd9F78P6UT6Z0=')).decode()
assert hashlib.sha256(VERIFIED_SOURCE_WORKER_TEXT.encode()).hexdigest()=='ee77cb475a9589ab21634c16b7316dfdec5abe95625fef29213335a0c28c0079'
VERIFIED_SOURCE_WORKER_PATH=WORKSPACE/'verified_source_worker.py';VERIFIED_SOURCE_WORKER_PATH.write_text(VERIFIED_SOURCE_WORKER_TEXT)
shutil.copyfile(VERIFIED_SOURCE_WORKER_PATH,SESSION/'verified_source_worker.py')
RENDERER_WORKER_TEXT=zlib.decompress(base64.b64decode('eNqVWlt32za2fvev4LgPIBuatjNNz6k9fHAdJdXUsX1sOdMzqoaLJiGJNUVyAFK22pX/Pt/GhRdJSTPuakwC4MbGvnz7Ah8eHo5lmcc1Tx3Bi5QLLhz+UnGRrXhRB85kyZ0qxlTtyKaq8oxLZ43ZeYYvlpmsS5Elce4seY6PHFk2IuHB4eHhQbaqSlE7saz9pKw2/jKWyzx79PU4noIVr+M0rmM/K2TFk9r/TZaFX0pfcF9upF+DB78WccIf4+TpYC7KFXipiYpjqN/i9eAg5XNHgrc6WpVpk3Ppaj68swMHP7KORS3DHNy6ggfzrEizmgtXsH9943y4eftwNXLc6cXRP+Oj30+Ofohmr34Nqo33a8F8TQcMBR8eribjq/H1yPMUUcHrRhTOH6tgIcqmck+9M714ugogSBevattp9up0Fqhn13OyuYP3v+W8cPW05/BcckcNaJ5nzrwUTuavnKxweNGsuIB67PJP5rTYvYqes3pZNnTqpqiHZ64F5yFkH0B5ktu588cy3YTTmVpC2xS0Ca0NaEZ/Sj/Ep4Ra6rhIuFv4ROlCSi5qz4mL1GH3Nw93l6Po7uE6yGSUZsL1GNGihU2hNy2Cmsva66jSjx7sscbuLv4R3V5MfiJC8yznoORDjzxkfB3nzFOsfZGzsTKGd7APzV0RaDsIQ7Yoy0XOgwQ2/sjOkrKos6LhX6Q2eqmEptOfCsBLw9WCyzjPP7cgmDdFosVV1yJ7bGr+xaU9stfxilv+txcEWYrDpCJbc7a7JMZWmFZm8JWHhDKzRaG3i4uN21tQb3FT683hKCOmjKYmRUOPsVhwmOS2fhVjSsGXpaJZu8xiSyR4FWeCee03pNwgrirymWLgWkm5qsgciNI8e4lWGXguFlFeJnGdgbSa+aA07SrDpn/8elPxCGcrBZcwdc/zmQWsI+U2R0V5JLgWls/4C0/AjvIqeobCIjxnRab3MAiQ1C/moNqFis59drxMGeyO1BW376AuovuWz7Up5bGUePE87ZPEgds/+J8er3c6jcFHPe7b8yn+9SG1BiLNq4KQCEBLguXrLOXE7ABJVrF44iI8tAp7FgDPKJFrd17mUOoxq7IXgFiAIebrZ+98dxVfI4hEyTIuina1HhPls/QO1WaxwhgbRhKFa5oBLwxPfYf91IUczbVjuYYq6Fsc26nKDPGKtlrwNGBGsogk4eGvxefPoY00mguYvpGJ4XP6B1Oj7AyE3QxCN2u1KDlcLpXsbJ6Xce2uvU8GwtdDCG8Jc+nNzIG/ce7jNXdqhNk4FzyG4ax4LBuB8KpWOvEcsUotyMuyCpzr0kRqsOdjHHawxAGkQ56pRfIIuaWxoFithRD0/coIF3aQI7Qa8fr61yslJmMpy1ikz4j9kbUwdyHiapkl1uehvbQUoaxFOxMAElymJ2B9zPOCvHxGsPXOLQbsWW+nhl/0mWYF1JzFKsZo6gqbHssyd+3XiLIiq1xPw5Yd/UvIirKwsFnWCu4k0WkTHtKWGpmyPF/D+SoOTmQ5r3uPJAl6fBbwSzYzMtJWRImLS6mJr8DPiIfm8X/c5GQUg+CKZIhA2137rC4pMWHemTnpOtAjrvcZAF/7lPW060ma626tiDPkExOgxEiIUriEF5gPoqgg64v0SuI1JDKKa++c/g10nhesniic6xcZTgTCE38BQ1H5pN40hXpVheorhSCymQOiFbFAP79iAZawdnGgJVXDwFwSV5A2q0q6OvwhI8Nm4WsfkbV8BqdF+C5GXuQb8YXmt+edl7K1XFD1FftaFVlBaFKKjWtkrRLGVfNbmZQBX+Q2Z8RjBC6ANc7o/ZXR1DpLuEkRMUjL/6/hYvNWT4x+mcCszgmo+tlT5qdDDzd0errGJ+EfjM73AvD41I7/m6iHMH5lBtgSSD3Y874mB8fGzL+G+Q6MQX08DLv19oDyTxWJ1Wo39U9e/nry5o1HmIP9IpAeXd+Pb67vzx14FiKKrLPEKYt84ys/eX/74CSEqPNMB9xgZwOcbooIU/NCqnAzC3VWkgIRU+5yskAZMqMw5m1bslL+4wYZocmDlTErFxrsxV8SXtXOSP0ihIfuMHam9lfni9ReYIAoYMrra0DaBAPPw+ydgZO6AXSzy5sPt1ejyegtfNyokRF9CZdH/QLoZ5Baq2swce7ESd0gEF1/HL8dXziYpdSLTAuSzHNs8u8mA5Azk7QjOqSRWeKaSsXXwcfXW0ZZ6s8bZJdalwjaMB7rueCgTaPiBiVM9FwSZoPh0ceLqwgJ2sXVePL/WKfiqmSfupxiX8FgWJiyk9NIzaPqAbDpXMHsP2U3D5Pbhwkkqzk9V4P3o3synXb0wFgmMd/Z4TfOz5xXCGx5v1SEBObZ4pgqv2OkMas4z35XAnUW8CHpO4ZLeCUCekbTFNQqoGKycZ55tljWsrNFkmbYHeV1FJPJQjVSHaddVwnkRS8hLQ9UseiyZMmTJ5UsRAoQu6Lm+PKn0eXPtzfj60l0N7q6mIw/jpg3PenIaZQL2f0tZB4qRCP9Svaqv8POrH/qTU93qOhf5qPD27ubyc3lzRV0T3UQOzukr3p7D7JEfa5eGmjPf0R5UY1EkG/nge3uPEfZzdNQ67Q9PZsdq5F9Qui4MLmaWkl1W0QVPpu5lioyNjU3+uV2dAnHinrk7n+6eP3mezYbZnSdPhws2Ergdg6uhdY7OFnUkUx4wY/IywRksOfghDO9WKxjwvJNtfFfVnnAVTE8yjm1PyZ4VmFi0q7XJyJ32B4kcVAuWyI50KFVDYvysZQNQh+bIQYrOUUeQpgs8zVKXRN0j5Wmc3msqMjuxOQLirvgHZ1ZkbQFM8THUEoRh/Mh+Gsurz9GH0aTC7DTGaA7nzKSE7ihyAOP4cUaLrOQlNIMsD1+Ds3i4xTlUnT65rvuK8WuOk7PIjRe3I3fj6+BR798uMLWIGPDwTb8Y2oA/njvYOfy4sPo7uIeFCDaBUS0zvgz1EnyrE8ivgFYFjC5IjUMaNHc/Hgzif5ONqY+ndv1v5FV/ZF90hVsptI/si731P9fb/ZqyhZIHVE9nUSC8AUnw6TQn51i2y9Mv2Zd7bZdPXa49F2kASwCxxFknomyUC02AqnOOncwdGDyPXJxpIsnxDserzSVzhU+vh4dYaMjnesfqXDxGRD4E7YfzT4Km7eY3ffpV5SXPepJtNUZ2BuC7i9H1yNjT9qp9CZwWODNHqvT3wJVFlkR59EqfuLhjjfSqJV5G35hnyr5oZyrU4JuuWlYU9Tcb8ll/G+/fXqmh60eyDeqbUqZFAU7aKhJAHEGz2gHqubK+VwmAJiCsjHDgNlkJ9fSu0xt1bmomqjlGDKhorR93/JinegMRLHN/NB/t0UUDg7e5r5Qoq6Y2vVBg6xHBo/UXoW61ZtxWD1vVY2iWidBlGbbhIjNOhmuQrmRpoenK0Q8DzLglarmbPq88pmFVpMtD/VRaTxedQB8bot+otwWFCwAvxHzULvADFGI6ppFYZRbBVSem5dhExhy0xmerWqjCqdWaG04gcOHxNjBTq5OM32bJiH38cEaeM8HfAYwQJjZUhykEZWPiMlrlU3hQ5DprdFybmsOTAYyI9EZqzILotYwtyuPRd5+bNYCF6HA+mVPjUL1VudKSl0I8Xn8CBizVBa5z9R5k0ZQFDRkvLNFHvTH+6Wwzmk2WmatvO0wDr27NmiqlKoze8CEmm6hKowXeVcZ7+FahSV68ofh1RAyXeYeJT3y39Da59J9Hfn7nV5Lakiq7dfoXknbtgl3GzlWNF6vEav9dsf1/sTBd8U9ZXaFrr2jBVC6PZKZg9l9ePg7stzo/dWO9bSE6MxtswuukYiyR2oPXGEBCjZL+fYBeev129Hd+Pq92aTffNn+noyRCDCvU9bBFpBa3jQOEqTmw2sTqGwQQ8jNkryUSH80RlB4j5JYKnjwqUbksva3OmvZ3DEzBAyqjDRaBU6axPsL7Tmk1ZfwuIzMHsW97nvBp+k1LmzBaglYez7HMucSVf8cp6IrNxS4VHC3KXjHlW3bac7aplkPdw2Te/uCHl0ncBnrnlzXc6OA8rXNwWHlQFw/QqZ5VnQHbqBD2saxG5hjCKSjIh10cijB1S1BI3QakP2YZM5jFyrbQYmBHDRDWUeFzA3FezVE5SoefuMU9qkLu4o3ziMnhvrVDBnBAPtVg8BSUDYCPKMdvZ2iC3NIwuMX3a2GU4T/c6KkSTeJepZTT3oGWX8P5t5lLzwFD0WypE6vqYYRQcsVSp7Nnlqrz5gqzExORwEJ9HuBxVoaNbpSdtbKkHIT6p6gIGNnrOarii6aocY2KaIwdWaimc+QxnBR0W20HocO+WNZPqGm1w00n5FV4lyYfBbU0xHsU89DSa0DgbasGab9dsDfDqrUn1Zm0faK1Ktply43EvbYhXB4AnzEFVPbQIIC2O3F/b2uL4S2JUWv5xOWjNp7HeeKk8iMRqgEtyPYIG3Y2+Zr0webN+D70x9++M7TfRoFDZutfGKQFpgL8ppuk0xKQG4XqTHUp3Ooo4ZNeQHd4rvebPClKZ+LZlVtqA4tqsE0ySJWala54jQ49U/a/46C1zN/euL3Bs1If2w22+1rKj5rXrlFFQDLhYg3rt7GTykYhxhXdzF/fe155/p0Vq1fc8ahdEBMxmtTe7eNr2M2J5dqtaf+TiEokDf6Pa703j2COyHjK8LG/oaltrb/xhAVPhh++897vlXxj42vqSt6MRn/eDViXa7Rpl3szD5RlqI2IfdXD0hSNXV2Zh4woitggjkOH9YtfmqvCgiSV5ksU+q6nvS2sr1XuufiHbpq+OL6GGkmq1JSZtmeTuXf8lzJU5R5Tu1EClZwOnVXSKLhcX5EV2wOYGROfUhM2FbtCvu4BsjsbYIqGHQl0UIc6FYN3Qad6+mvvDxRf8yBxIy2D2j7SF11mmsySTcee7G0juVTb1C9EsRam7h7uL6mXEeDYu9uqrvi1OGbBplNPrzdqkRD/dZh56L8nRdtGPB0QaS62TsByvy1TyCX8es335sb5QCyV40gL1jylzRbcLrmCsOdHdRHvcbSduLRlqTYQsRiw7YKvraxOTzAzmdbLa/P8G6+GtZ/nznB9hbtWRCD782cY+Zs0DVXvFkt1RWBNt5eJO4ng1rl8NP2tovsgSzGVjm9a7BeIBk0Pe3dwuBCYu8fUNnriYH0uyrE83cZo/yWDeVqUSLcUx+f77HSnbLO2Gub3w6re0H9D3ahL2NstnjWLffneSOXPff7omAVa9tC1R4W9m+Jdg61dWVFEh8S+ZPM3/bU1FXXj1i2c93VS3X2MXfx480dsbbVZsWX/s9881gi3R8TyIimqk2Jw95djK/oNOr6rL086/7qLmyfAoWRdYR5a1k7wUzzhdQzjyvkuu1fRsz2Id2RQcGD4V9kDCxB33BF5rwWtejFO9jS/z/UUofUn57ZVQY47YuNczsWYcKrXndwAAHargAsAmU95fkRO1NhoddMV/BCBXMsFuvp6RASIaT/AHTQruM=')).decode()
assert hashlib.sha256(RENDERER_WORKER_TEXT.encode()).hexdigest()=='3457682def3bd31babe706b90c9e08320cbd479a050f3e3e562042a3dc26867b'
RENDERER_WORKER_PATH=WORKSPACE/'renderer_worker.py';RENDERER_WORKER_PATH.write_text(RENDERER_WORKER_TEXT)
shutil.copyfile(RENDERER_WORKER_PATH,SESSION/'renderer_worker.py')
VERIFIED_LIBRARY_WORKER_TEXT=zlib.decompress(base64.b64decode('eNptVVFv2zYQfvevIPxCCVCVNcD24EAPWZ0GATRn6LruwTAIWjzbnClS49FOjLT/fUdSdrKuQYCIx+N3d999d5lOp62TiknL4Flj0HbLjOukYXf3LTuCVc4zo9de+hPTlkmmNA4O5doAG7zrALFmC8e2Xg473SHT/eB8qKfT6SR/si6cBsBqJ3FHSNXf6GzlsMITVkH3UAUvO1jLbj/ZeNezQYboNwKx3+k4mSjYUHgcoAtiTKeIjuVswuhnaKJbttxIRKCHQ61RbLSBoqwY/wMMvQU1VncpCVmvEansmieky2Mre6gxSB/wSRM2pyfEibBHrbSs0fEI+2jN6T/kLb48zB9uf8ReL09sDRRfKlBjtPGuyRTVH+ZtW2DwxVBWvVPQbCHIQGeii3/63M7F4vEvXl2XX/930T5+uG159VNZJmCwgWDPXmOYigsBWyN+k9ryauEslG9rTm8iI9YFFm+pvjZle6nBUJeQhR2w+/bLYv62zPx6cNqGsbheDpj7wq+iVK4QzOYqWnmOm6lokNoMqngx2hLjg9GhKJfv3q/YJrFnIQovPqs9SCUCPJNDdoy3WJRMbxi/4tEv+UsbFX0q9mdLbdwTeHKMkMm65LmRvIqNJVboY2uOVvFV+S0BpoiprzECGAS2XKW8PYSDt+yFkzzCAfmMt4+389tf27uMFqnis7GP3AM6cwQlojpHcz0ai7KsEiRjHHfy+udf+Gyckzqfky9VvT6FWGlZ7+BZ6S0gcVDx2M3Mv0j8i8Q/n332B6BUEr/iPJopAco2m1/jdm4AqkGdSPK6e+113AuO9H2TBEEddBtGv7HnCo66I56PUhu51tSJEyNmR/F7ygg8TQP/lie3J8EVHv45UNrjxG6cIZ8sj/Fmyd0hDIdAHbjJ13W/V5rokoQYsElVpY4It0+n8iZNKIkobpJ6AL8RnTvYEJs99goPJjQvXCs+uwSiw6riQeL+jTEdyXzp6qc/F4uHxT1VEZGI3llGqw+DkgGK7xfSBemsgdV5Gp87GAK7S3+0ozWK0fYdXI7bJDGJj7cP7d2cV+C9800UDT0oX3dlc/mqSdO9DILui/Jt0UsOhjRMCkDonFXIVz+i6d1IYXpZZOKv+JPze/BizDDubF7WT14HyAMYLbU69AMW2afS1HQbmutKGpo2YaVtPkqamktOaWiy82RCAyZE3LFCNA3tpSgRIfgsSSWBR/1hkQRC/ypq6bfH5ftV+XYJEPS/GVZPcQ==')).decode()
assert hashlib.sha256(VERIFIED_LIBRARY_WORKER_TEXT.encode()).hexdigest()=='e1e854a82dbd5a3445d43d8e41966164cd991e321117f2321e22fb8a1f4b87d6'
VERIFIED_LIBRARY_WORKER_PATH=WORKSPACE/'verified_library_worker.py';VERIFIED_LIBRARY_WORKER_PATH.write_text(VERIFIED_LIBRARY_WORKER_TEXT)
shutil.copyfile(VERIFIED_LIBRARY_WORKER_PATH,SESSION/'verified_library_worker.py')
VERIFIED_VERIFIED_PAIRED_V4_TEXT=zlib.decompress(base64.b64decode('...')).decode()
assert hashlib.sha256(VERIFIED_VERIFIED_PAIRED_V4_TEXT.encode()).hexdigest()=='48d0045520329ab21c8e7cc70ec38cbc26b8a1de892c3c0ea62a085fc6e875e4'
VERIFIED_VERIFIED_PAIRED_V4_PATH=WORKSPACE/'verified_verified_paired_v4.py';VERIFIED_VERIFIED_PAIRED_V4_PATH.write_text(VERIFIED_VERIFIED_PAIRED_V4_TEXT)
shutil.copyfile(VERIFIED_VERIFIED_PAIRED_V4_PATH,SESSION/'verified_verified_paired_v4.py')
FROZEN_PATH=VERIFIED_VERIFIED_PAIRED_V4_PATH
LIBRARY_WORKER_PATH=VERIFIED_LIBRARY_WORKER_PATH
# Reuse the exact verified installer. Rendering imports happen only in fresh subprocesses.
frozen_parts=dict((m.split('\n',1)[0],m.split('\n',1)[1]) for m in FROZEN_PATH.read_text().split('# MODULE ')[1:])
installer=next(n for n in ast.parse(frozen_parts['01_setup.py']).body if isinstance(n,ast.FunctionDef) and n.name=='ensure_dependencies')
exec(compile(ast.Module(body=[installer],type_ignores=[]),'verified-dependency-installer','exec'),globals())
ensure_dependencies()
requirements={'cv2':'opencv-python-headless','argcomplete':'argcomplete','engineering_notation':'engineering-notation',
    'screeninfo':'screeninfo','easygui':'easygui','plyer':'plyer','dv_processing':'dv-processing','scipy':'scipy','tqdm':'tqdm'}
missing=[package for module,package in requirements.items() if importlib.util.find_spec(module) is None]
if missing:subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
importlib.invalidate_caches()
exec(compile(COMMON_TEXT,str(COMMON_PATH),'exec'),globals())
exec(compile(D200_CONTRACT_TEXT,str(D200_CONTRACT_PATH),'exec'),globals())
exec(compile(SOURCE_HELPERS_TEXT,str(SOURCE_HELPERS_PATH),'exec'),globals())
exec(compile(D200_SOURCE_CONTROLLER_TEXT,str(D200_SOURCE_CONTROLLER_PATH),'exec'),globals())
atomic_json(SESSION/'code_identity.json',{'version':CODE_VERSION,'build_sha256':BUILD_SHA,'stage':STAGE})


## 3. Automatic execution, resume and evidence publication


In [ ]:
SUMMARY=generate_d200()
